# Kaggle job: freeze

freeze

Inputs are resolved from the declared resource graph. No raw dataset re-upload is required.

# Prepare existing resources for the fixed multi-view evaluation

This reusable entry point executes one configured preparation operation. The launch kit binds existing Kaggle datasets and upstream outputs automatically; it does not re-upload the raw dataset. Models and the two selected scoring methods stay fixed. All scientific output, private checkpoints and reports remain separate.

Operations are explicit: method freeze, private RGB index, model-only RGB release, neutral reference geometry/correspondence, missing-feature and cache shards, cache merge, and post-prediction truth conversion. Only truth conversion opens anomaly outcomes, after complete model predictions verify. The trusted raw resource preparer may read mixed source metadata but exports only whitelisted neutral fields.

In [1]:
from pathlib import Path
import os
import json
RUNTIME_PROFILE = None
NOTEBOOK_RELATIVE = 'notebooks/input-preparation/current-kaggle-preparation.ipynb'
try:
    from pathlib import Path, PurePosixPath
    import hashlib
    import importlib.util
    import json
    import os
    import shutil
    import stat
    import sys
    import tempfile
    import zipfile

    # Bounded input lookup supports flat, namespaced and automatically expanded mounts.
    def _children(path):
        if not path.is_dir():
            return []
        children = sorted(p for p in path.iterdir() if p.is_dir())
        if len(children) > 250:
            raise ValueError('Source discovery bound exceeded; set VPI_CODE_ROOT explicitly')
        return children

    def _source_candidates():
        explicit = os.getenv('VPI_CODE_ROOT')
        if explicit:
            return [Path(explicit).expanduser()]
        mount = Path('/kaggle/input')
        roots = [Path.cwd(), *Path.cwd().parents]
        roots += [p for p in _children(mount) if p.name not in {'datasets', 'notebooks', 'models'}]
        for category, depth in (('datasets', 2), ('notebooks', 2), ('models', 4)):
            current = [mount / category]
            for _ in range(depth):
                current = [child for parent in current for child in _children(parent)]
                if len(current) > 1000:
                    raise ValueError('Source mount inventory exceeded its bound')
            for parent in current:
                roots += _children(parent / 'versions') if (parent / 'versions').is_dir() else [parent]
        return [root / suffix for root in roots for suffix in ('', 'code', 'repository-files', 'source')]

    def _archive_source():
        configured = os.getenv('VPI_SOURCE_ARCHIVE')
        if not configured:
            raise FileNotFoundError('Mount the prepared source package or set VPI_CODE_ROOT explicitly')
        archive = Path(configured)
        expected = os.getenv('VPI_SOURCE_ARCHIVE_SHA256')
        if not expected or archive.stat().st_size > 256 * 1024 * 1024 or hashlib.sha256(archive.read_bytes()).hexdigest() != expected:
            raise ValueError('Explicit source archive checksum or size check failed')
        destination = Path(os.getenv('VPI_SOURCE_DEST', '/kaggle/working/current-source')).resolve()
        if Path('/kaggle').is_dir() and not destination.is_relative_to(Path('/kaggle/working')):
            raise ValueError('Extracted source must be under /kaggle/working')
        if destination.exists():
            raise ValueError('Source destination exists; verify it with VPI_CODE_ROOT instead of overwriting')
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = Path(tempfile.mkdtemp(prefix='current-source.partial-', dir=destination.parent))
        try:
            with zipfile.ZipFile(archive) as bundle:
                infos = bundle.infolist()
                names = [i.filename for i in infos]
                if len(names) != len(set(names)) or sum(i.file_size for i in infos) > 256 * 1024 * 1024 or bundle.testzip():
                    raise ValueError('Source archive inventory, size or CRC is invalid')
                for info in infos:
                    name = info.filename
                    path = PurePosixPath(name)
                    if info.is_dir() or path.is_absolute() or '\\' in name or any(p in {'', '.', '..'} for p in name.split('/')) or stat.S_ISLNK(info.external_attr >> 16):
                        raise ValueError('Unsafe source archive member')
                    output = temporary / name
                    output.parent.mkdir(parents=True, exist_ok=True)
                    output.write_bytes(bundle.read(info))
            os.replace(temporary, destination)
        except BaseException:
            shutil.rmtree(temporary)
            raise
        return destination

    _matches = sorted({p.resolve() for p in _source_candidates() if (p / 'prepared_source_manifest.json').is_file()})
    if len(_matches) > 1:
        raise ValueError('Multiple prepared source roots found; set VPI_CODE_ROOT explicitly')
    CODE_ROOT = _matches[0] if _matches else _archive_source()
    _manifest = json.loads((CODE_ROOT / 'prepared_source_manifest.json').read_text())
    # Verify the bootstrap itself before executing any code from the mounted package.
    _helper = CODE_ROOT / 'scripts/current_notebook_runtime.py'
    if _helper.is_symlink() or hashlib.sha256(_helper.read_bytes()).hexdigest() != _manifest.get('files', {}).get('scripts/current_notebook_runtime.py'):
        raise ValueError('Prepared bootstrap source checksum differs')
    _spec = importlib.util.spec_from_file_location('current_notebook_runtime', _helper)
    _runtime = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(_runtime)
    _launch_helper = CODE_ROOT / 'scripts/current_launch.py'
    if hashlib.sha256(_launch_helper.read_bytes()).hexdigest() != _manifest['files']['scripts/current_launch.py']:
        raise ValueError('Launch helper checksum differs')
    sys.path.insert(0,str(CODE_ROOT/'scripts'))
    from current_launch import configure_runtime, resolve_job, complete_job
    LAUNCH_PLAN = {'core_views': [2, 3, 5], 'diagnostic_views': [1], 'jobs': [{'attach_resources': [], 'config': {'action': 'prepare', 'build_wheelhouse': True, 'download_models': False, 'output_root': '/kaggle/working/setup-cpu', 'profile': 'cpu', 'target': '/tmp/vpi-runtime-setup-cpu', 'wheelhouse': '/kaggle/working/setup-cpu/wheelhouse'}, 'depends_on': [], 'id': 'setup-cpu', 'notebook': 'notebooks/input-preparation/current-kaggle-runtime-setup.ipynb', 'purpose': 'Setup only: resolve/install pinned Linux dependencies and record actual runtime; no dataset experiment', 'runtime': 'cpu', 'setup_job': True}, {'attach_resources': [], 'config': {'action': 'prepare', 'build_wheelhouse': True, 'download_models': True, 'model_asset_root': '/kaggle/working/setup-dino/model-assets', 'models_only': 'all', 'output_root': '/kaggle/working/setup-dino', 'profile': 'dino', 'target': '/tmp/vpi-runtime-setup-dino', 'wheelhouse': '/kaggle/working/setup-dino/wheelhouse'}, 'depends_on': [], 'id': 'setup-dino', 'notebook': 'notebooks/input-preparation/current-kaggle-runtime-setup.ipynb', 'purpose': 'Setup only: resolve/install pinned Linux dependencies and record actual runtime; no dataset experiment', 'runtime': 'cpu', 'setup_job': True}, {'attach_resources': [], 'config': {'action': 'prepare', 'build_wheelhouse': True, 'download_models': False, 'output_root': '/kaggle/working/setup-regions', 'profile': 'regions', 'target': '/tmp/vpi-runtime-setup-regions', 'wheelhouse': '/kaggle/working/setup-regions/wheelhouse'}, 'depends_on': [], 'id': 'setup-regions', 'notebook': 'notebooks/input-preparation/current-kaggle-runtime-setup.ipynb', 'purpose': 'Setup only: resolve/install pinned Linux dependencies and record actual runtime; no dataset experiment', 'runtime': 'cpu', 'setup_job': True}, {'attach_resources': [], 'config': {'action': 'prepare', 'build_wheelhouse': True, 'download_models': False, 'output_root': '/kaggle/working/setup-tracking', 'profile': 'tracking', 'target': '/tmp/vpi-runtime-setup-tracking', 'wheelhouse': '/kaggle/working/setup-tracking/wheelhouse'}, 'depends_on': [], 'id': 'setup-tracking', 'notebook': 'notebooks/input-preparation/current-kaggle-runtime-setup.ipynb', 'purpose': 'Setup only: resolve/install pinned Linux dependencies and record actual runtime; no dataset experiment', 'runtime': 'cpu', 'setup_job': True}, {'config': {'action': 'freeze', 'output_root': {'$output': 'freeze'}, 'reference_root': {'$resource': 'support', 'path': 'method-reference'}}, 'depends_on': ['setup-cpu'], 'id': 'freeze', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'freeze', 'runtime': 'cpu'}, {'config': {'action': 'rgb_index', 'administrative_root': {'$resource': 'administrative'}, 'output_root': {'$output': 'development-index'}, 'raw_root': {'$resource': 'raw'}, 'role': 'development', 'selection_root': {'$resource': 'selection'}}, 'depends_on': ['setup-cpu'], 'id': 'development-index', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'rgb index', 'runtime': 'cpu'}, {'config': {'action': 'raw_geometry', 'administrative_root': {'$resource': 'administrative'}, 'output_root': {'$output': 'development-geometry'}, 'raw_root': {'$resource': 'raw'}, 'rgb_index_root': {'$resource': 'development-index'}, 'role': 'development', 'selection_root': {'$resource': 'selection'}}, 'depends_on': ['development-index', 'setup-cpu'], 'id': 'development-geometry', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'raw geometry', 'runtime': 'cpu'}, {'config': {'action': 'rgb_index', 'administrative_root': {'$resource': 'administrative'}, 'output_root': {'$output': 'calibration-index'}, 'raw_root': {'$resource': 'raw'}, 'role': 'calibration', 'selection_root': {'$resource': 'selection'}}, 'depends_on': ['setup-cpu'], 'id': 'calibration-index', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'rgb index', 'runtime': 'cpu'}, {'config': {'action': 'raw_geometry', 'administrative_root': {'$resource': 'administrative'}, 'output_root': {'$output': 'calibration-geometry'}, 'raw_root': {'$resource': 'raw'}, 'rgb_index_root': {'$resource': 'calibration-index'}, 'role': 'calibration', 'selection_root': {'$resource': 'selection'}}, 'depends_on': ['calibration-index', 'setup-cpu'], 'id': 'calibration-geometry', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'raw geometry', 'runtime': 'cpu'}, {'config': {'action': 'rgb_index', 'administrative_root': {'$resource': 'administrative'}, 'output_root': {'$output': 'internal_lock-index'}, 'raw_root': {'$resource': 'raw'}, 'role': 'internal_lock', 'selection_root': {'$resource': 'selection'}}, 'depends_on': ['setup-cpu'], 'id': 'internal_lock-index', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'rgb index', 'runtime': 'cpu'}, {'config': {'action': 'raw_geometry', 'administrative_root': {'$resource': 'administrative'}, 'output_root': {'$output': 'internal_lock-geometry'}, 'raw_root': {'$resource': 'raw'}, 'rgb_index_root': {'$resource': 'internal_lock-index'}, 'role': 'internal_lock', 'selection_root': {'$resource': 'selection'}}, 'depends_on': ['internal_lock-index', 'setup-cpu'], 'id': 'internal_lock-geometry', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'raw geometry', 'runtime': 'cpu'}, {'config': {'action': 'rgb_index', 'output_root': {'$output': 'seen-index'}, 'raw_root': {'$resource': 'raw'}, 'role': 'seen'}, 'depends_on': ['setup-cpu'], 'id': 'seen-index', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'rgb index', 'runtime': 'cpu'}, {'config': {'action': 'raw_geometry', 'output_root': {'$output': 'seen-geometry'}, 'raw_root': {'$resource': 'raw'}, 'rgb_index_root': {'$resource': 'seen-index'}, 'role': 'seen'}, 'depends_on': ['seen-index', 'setup-cpu'], 'id': 'seen-geometry', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'raw geometry', 'runtime': 'cpu'}, {'config': {'action': 'rgb_index', 'output_root': {'$output': 'unseen-index'}, 'raw_root': {'$resource': 'raw'}, 'role': 'unseen'}, 'depends_on': ['setup-cpu'], 'id': 'unseen-index', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'rgb index', 'runtime': 'cpu'}, {'config': {'action': 'raw_geometry', 'output_root': {'$output': 'unseen-geometry'}, 'raw_root': {'$resource': 'raw'}, 'rgb_index_root': {'$resource': 'unseen-index'}, 'role': 'unseen'}, 'depends_on': ['setup-cpu', 'unseen-index'], 'id': 'unseen-geometry', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'raw geometry', 'runtime': 'cpu'}, {'config': {'action': 'official_exposure', 'index_roots': {'seen': {'$resource': 'seen-geometry'}, 'unseen': {'$resource': 'unseen-geometry'}}, 'output_root': {'$output': 'official-exposure'}, 'policy': 'exposure_qualified_fixed_methods'}, 'depends_on': ['seen-geometry', 'setup-cpu', 'unseen-geometry'], 'id': 'official-exposure', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'official exposure', 'runtime': 'cpu'}, {'config': {'action': 'neutral', 'components': {'toysad8k-development-confirm-300-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-confirm-300-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-confirm'}}, 'toysad8k-development-remainder-600-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-remainder-600-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-remainder'}}, 'toysad8k-development-screen-100-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-screen-100-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-screen'}}}, 'expected_region_lock_sha256': {'$resource': 'development-regions', 'path': 'full-development-lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'full_development_lock': {'$resource': 'development-regions', 'path': 'full-development-lock.json'}, 'geometry_root': {'$resource': 'development-geometry'}, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': 'development-neutral'}, 'region_format': 'native-development-components', 'role': 'development'}, 'depends_on': ['development-geometry', 'freeze', 'setup-cpu'], 'id': 'development-neutral', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'neutral', 'runtime': 'cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'components': {'toysad8k-development-confirm-300-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-confirm-300-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-confirm'}}, 'toysad8k-development-remainder-600-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-remainder-600-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-remainder'}}, 'toysad8k-development-screen-100-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-screen-100-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-screen'}}}, 'expected_region_lock_sha256': {'$resource': 'development-regions', 'path': 'full-development-lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'development-cache-00-features'}, 'feature_roots': [{'$resource': 'reuse100'}], 'full_development_lock': {'$resource': 'development-regions', 'path': 'full-development-lock.json'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'development-neutral'}, 'output_root': {'$output': 'development-cache-00'}, 'region_format': 'native-development-components', 'role': 'development', 'shard_count': 10, 'shard_index': 0}, 'depends_on': ['development-neutral', 'freeze', 'setup-dino'], 'id': 'development-cache-00', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'components': {'toysad8k-development-confirm-300-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-confirm-300-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-confirm'}}, 'toysad8k-development-remainder-600-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-remainder-600-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-remainder'}}, 'toysad8k-development-screen-100-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-screen-100-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-screen'}}}, 'expected_region_lock_sha256': {'$resource': 'development-regions', 'path': 'full-development-lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'development-cache-01-features'}, 'feature_roots': [{'$resource': 'reuse100'}], 'full_development_lock': {'$resource': 'development-regions', 'path': 'full-development-lock.json'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'development-neutral'}, 'output_root': {'$output': 'development-cache-01'}, 'region_format': 'native-development-components', 'role': 'development', 'shard_count': 10, 'shard_index': 1}, 'depends_on': ['development-neutral', 'freeze', 'setup-dino'], 'id': 'development-cache-01', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'components': {'toysad8k-development-confirm-300-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-confirm-300-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-confirm'}}, 'toysad8k-development-remainder-600-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-remainder-600-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-remainder'}}, 'toysad8k-development-screen-100-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-screen-100-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-screen'}}}, 'expected_region_lock_sha256': {'$resource': 'development-regions', 'path': 'full-development-lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'development-cache-02-features'}, 'feature_roots': [{'$resource': 'reuse100'}], 'full_development_lock': {'$resource': 'development-regions', 'path': 'full-development-lock.json'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'development-neutral'}, 'output_root': {'$output': 'development-cache-02'}, 'region_format': 'native-development-components', 'role': 'development', 'shard_count': 10, 'shard_index': 2}, 'depends_on': ['development-neutral', 'freeze', 'setup-dino'], 'id': 'development-cache-02', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'components': {'toysad8k-development-confirm-300-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-confirm-300-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-confirm'}}, 'toysad8k-development-remainder-600-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-remainder-600-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-remainder'}}, 'toysad8k-development-screen-100-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-screen-100-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-screen'}}}, 'expected_region_lock_sha256': {'$resource': 'development-regions', 'path': 'full-development-lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'development-cache-03-features'}, 'feature_roots': [{'$resource': 'reuse100'}], 'full_development_lock': {'$resource': 'development-regions', 'path': 'full-development-lock.json'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'development-neutral'}, 'output_root': {'$output': 'development-cache-03'}, 'region_format': 'native-development-components', 'role': 'development', 'shard_count': 10, 'shard_index': 3}, 'depends_on': ['development-neutral', 'freeze', 'setup-dino'], 'id': 'development-cache-03', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'components': {'toysad8k-development-confirm-300-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-confirm-300-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-confirm'}}, 'toysad8k-development-remainder-600-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-remainder-600-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-remainder'}}, 'toysad8k-development-screen-100-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-screen-100-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-screen'}}}, 'expected_region_lock_sha256': {'$resource': 'development-regions', 'path': 'full-development-lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'development-cache-04-features'}, 'feature_roots': [{'$resource': 'reuse100'}], 'full_development_lock': {'$resource': 'development-regions', 'path': 'full-development-lock.json'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'development-neutral'}, 'output_root': {'$output': 'development-cache-04'}, 'region_format': 'native-development-components', 'role': 'development', 'shard_count': 10, 'shard_index': 4}, 'depends_on': ['development-neutral', 'freeze', 'setup-dino'], 'id': 'development-cache-04', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'components': {'toysad8k-development-confirm-300-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-confirm-300-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-confirm'}}, 'toysad8k-development-remainder-600-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-remainder-600-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-remainder'}}, 'toysad8k-development-screen-100-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-screen-100-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-screen'}}}, 'expected_region_lock_sha256': {'$resource': 'development-regions', 'path': 'full-development-lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'development-cache-05-features'}, 'feature_roots': [{'$resource': 'reuse100'}], 'full_development_lock': {'$resource': 'development-regions', 'path': 'full-development-lock.json'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'development-neutral'}, 'output_root': {'$output': 'development-cache-05'}, 'region_format': 'native-development-components', 'role': 'development', 'shard_count': 10, 'shard_index': 5}, 'depends_on': ['development-neutral', 'freeze', 'setup-dino'], 'id': 'development-cache-05', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'components': {'toysad8k-development-confirm-300-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-confirm-300-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-confirm'}}, 'toysad8k-development-remainder-600-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-remainder-600-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-remainder'}}, 'toysad8k-development-screen-100-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-screen-100-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-screen'}}}, 'expected_region_lock_sha256': {'$resource': 'development-regions', 'path': 'full-development-lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'development-cache-06-features'}, 'feature_roots': [{'$resource': 'reuse100'}], 'full_development_lock': {'$resource': 'development-regions', 'path': 'full-development-lock.json'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'development-neutral'}, 'output_root': {'$output': 'development-cache-06'}, 'region_format': 'native-development-components', 'role': 'development', 'shard_count': 10, 'shard_index': 6}, 'depends_on': ['development-neutral', 'freeze', 'setup-dino'], 'id': 'development-cache-06', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'components': {'toysad8k-development-confirm-300-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-confirm-300-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-confirm'}}, 'toysad8k-development-remainder-600-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-remainder-600-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-remainder'}}, 'toysad8k-development-screen-100-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-screen-100-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-screen'}}}, 'expected_region_lock_sha256': {'$resource': 'development-regions', 'path': 'full-development-lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'development-cache-07-features'}, 'feature_roots': [{'$resource': 'reuse100'}], 'full_development_lock': {'$resource': 'development-regions', 'path': 'full-development-lock.json'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'development-neutral'}, 'output_root': {'$output': 'development-cache-07'}, 'region_format': 'native-development-components', 'role': 'development', 'shard_count': 10, 'shard_index': 7}, 'depends_on': ['development-neutral', 'freeze', 'setup-dino'], 'id': 'development-cache-07', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'components': {'toysad8k-development-confirm-300-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-confirm-300-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-confirm'}}, 'toysad8k-development-remainder-600-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-remainder-600-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-remainder'}}, 'toysad8k-development-screen-100-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-screen-100-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-screen'}}}, 'expected_region_lock_sha256': {'$resource': 'development-regions', 'path': 'full-development-lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'development-cache-08-features'}, 'feature_roots': [{'$resource': 'reuse100'}], 'full_development_lock': {'$resource': 'development-regions', 'path': 'full-development-lock.json'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'development-neutral'}, 'output_root': {'$output': 'development-cache-08'}, 'region_format': 'native-development-components', 'role': 'development', 'shard_count': 10, 'shard_index': 8}, 'depends_on': ['development-neutral', 'freeze', 'setup-dino'], 'id': 'development-cache-08', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'components': {'toysad8k-development-confirm-300-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-confirm-300-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-confirm'}}, 'toysad8k-development-remainder-600-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-remainder-600-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-remainder'}}, 'toysad8k-development-screen-100-v1': {'region_root': {'$resource': 'development-regions', 'path': 'toysad8k-development-screen-100-v1/model-safe'}, 'rgb_root': {'$resource': 'rgb-screen'}}}, 'expected_region_lock_sha256': {'$resource': 'development-regions', 'path': 'full-development-lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'development-cache-09-features'}, 'feature_roots': [{'$resource': 'reuse100'}], 'full_development_lock': {'$resource': 'development-regions', 'path': 'full-development-lock.json'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'development-neutral'}, 'output_root': {'$output': 'development-cache-09'}, 'region_format': 'native-development-components', 'role': 'development', 'shard_count': 10, 'shard_index': 9}, 'depends_on': ['development-neutral', 'freeze', 'setup-dino'], 'id': 'development-cache-09', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'config': {'action': 'cache_merge', 'cache_shards': [{'$resource': 'development-cache-00'}, {'$resource': 'development-cache-01'}, {'$resource': 'development-cache-02'}, {'$resource': 'development-cache-03'}, {'$resource': 'development-cache-04'}, {'$resource': 'development-cache-05'}, {'$resource': 'development-cache-06'}, {'$resource': 'development-cache-07'}, {'$resource': 'development-cache-08'}, {'$resource': 'development-cache-09'}], 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'maximum_output_bytes': 19327352832, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': 'development-cache'}, 'role': 'development'}, 'depends_on': ['development-cache-00', 'development-cache-01', 'development-cache-02', 'development-cache-03', 'development-cache-04', 'development-cache-05', 'development-cache-06', 'development-cache-07', 'development-cache-08', 'development-cache-09', 'freeze', 'setup-cpu'], 'id': 'development-cache', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'cache merge', 'runtime': 'cpu'}, {'config': {'action': 'dev-predict', 'cache_root': {'$resource': 'development-cache'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': 'development-predict'}, 'role': 'development'}, 'depends_on': ['development-cache', 'freeze', 'setup-cpu'], 'id': 'development-predict', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'dev-predict', 'runtime': 'cpu'}, {'config': {'action': 'raw_truth', 'administrative_root': {'$resource': 'administrative'}, 'cache_root': {'$resource': 'development-cache'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'geometry_root': {'$resource': 'development-geometry'}, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': 'development-truth'}, 'prediction_root': {'$resource': 'development-predict'}, 'raw_root': {'$resource': 'raw'}, 'rgb_index_root': {'$resource': 'development-index'}, 'role': 'development'}, 'depends_on': ['development-cache', 'development-geometry', 'development-index', 'development-predict', 'freeze', 'setup-cpu'], 'id': 'development-truth', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'raw truth', 'runtime': 'cpu'}, {'attach_resources': ['runtime-tracking', 'wheels-tracking'], 'config': {'action': 'dev-evaluate', 'cache_root': {'$resource': 'development-cache'}, 'evaluator_root': {'$resource': 'development-truth'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': 'development-evaluate'}, 'prediction_root': {'$resource': 'development-predict'}, 'publish_tracking': True, 'report_root': {'$output': 'development-evaluate-report'}, 'role': 'development'}, 'depends_on': ['development-cache', 'development-predict', 'development-truth', 'freeze', 'setup-cpu', 'setup-tracking'], 'id': 'development-evaluate', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'dev-evaluate', 'runtime': 'cpu'}, {'config': {'action': 'rgb_release', 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'maximum_new_scenes': 600, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': 'calibration-rgb'}, 'raw_root': {'$resource': 'raw'}, 'rgb_index_root': {'$resource': 'calibration-index'}, 'role': 'calibration'}, 'depends_on': ['calibration-geometry', 'calibration-index', 'development-evaluate', 'freeze', 'setup-cpu'], 'id': 'calibration-rgb', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'rgb release', 'runtime': 'cpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': 'calibration-regions-00'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'role': 'calibration', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 6, 'shard_index': 0}, 'depends_on': ['calibration-rgb', 'freeze', 'setup-dino', 'setup-regions'], 'id': 'calibration-regions-00', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': 'calibration-regions-01'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'role': 'calibration', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 6, 'shard_index': 1}, 'depends_on': ['calibration-rgb', 'freeze', 'setup-dino', 'setup-regions'], 'id': 'calibration-regions-01', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': 'calibration-regions-02'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'role': 'calibration', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 6, 'shard_index': 2}, 'depends_on': ['calibration-rgb', 'freeze', 'setup-dino', 'setup-regions'], 'id': 'calibration-regions-02', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': 'calibration-regions-03'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'role': 'calibration', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 6, 'shard_index': 3}, 'depends_on': ['calibration-rgb', 'freeze', 'setup-dino', 'setup-regions'], 'id': 'calibration-regions-03', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': 'calibration-regions-04'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'role': 'calibration', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 6, 'shard_index': 4}, 'depends_on': ['calibration-rgb', 'freeze', 'setup-dino', 'setup-regions'], 'id': 'calibration-regions-04', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': 'calibration-regions-05'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'role': 'calibration', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 6, 'shard_index': 5}, 'depends_on': ['calibration-rgb', 'freeze', 'setup-dino', 'setup-regions'], 'id': 'calibration-regions-05', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'region_merge', 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': 'calibration-regions'}, 'region_shards': [{'$resource': 'calibration-regions-00'}, {'$resource': 'calibration-regions-01'}, {'$resource': 'calibration-regions-02'}, {'$resource': 'calibration-regions-03'}, {'$resource': 'calibration-regions-04'}, {'$resource': 'calibration-regions-05'}], 'rgb_root': {'$resource': 'calibration-rgb'}, 'role': 'calibration'}, 'depends_on': ['calibration-regions-00', 'calibration-regions-01', 'calibration-regions-02', 'calibration-regions-03', 'calibration-regions-04', 'calibration-regions-05', 'calibration-rgb', 'freeze', 'setup-cpu'], 'id': 'calibration-regions', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'region merge', 'runtime': 'cpu'}, {'config': {'action': 'neutral', 'expected_region_lock_sha256': {'$resource': 'calibration-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'geometry_root': {'$resource': 'calibration-geometry'}, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': 'calibration-neutral'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'calibration-regions'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'role': 'calibration'}, 'depends_on': ['calibration-geometry', 'calibration-regions', 'calibration-rgb', 'freeze', 'setup-cpu'], 'id': 'calibration-neutral', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'neutral', 'runtime': 'cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'expected_region_lock_sha256': {'$resource': 'calibration-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'calibration-cache-00-features'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'calibration-neutral'}, 'output_root': {'$output': 'calibration-cache-00'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'calibration-regions'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'role': 'calibration', 'shard_count': 6, 'shard_index': 0}, 'depends_on': ['calibration-neutral', 'calibration-regions', 'calibration-rgb', 'freeze', 'setup-dino'], 'id': 'calibration-cache-00', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'expected_region_lock_sha256': {'$resource': 'calibration-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'calibration-cache-01-features'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'calibration-neutral'}, 'output_root': {'$output': 'calibration-cache-01'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'calibration-regions'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'role': 'calibration', 'shard_count': 6, 'shard_index': 1}, 'depends_on': ['calibration-neutral', 'calibration-regions', 'calibration-rgb', 'freeze', 'setup-dino'], 'id': 'calibration-cache-01', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'expected_region_lock_sha256': {'$resource': 'calibration-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'calibration-cache-02-features'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'calibration-neutral'}, 'output_root': {'$output': 'calibration-cache-02'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'calibration-regions'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'role': 'calibration', 'shard_count': 6, 'shard_index': 2}, 'depends_on': ['calibration-neutral', 'calibration-regions', 'calibration-rgb', 'freeze', 'setup-dino'], 'id': 'calibration-cache-02', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'expected_region_lock_sha256': {'$resource': 'calibration-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'calibration-cache-03-features'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'calibration-neutral'}, 'output_root': {'$output': 'calibration-cache-03'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'calibration-regions'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'role': 'calibration', 'shard_count': 6, 'shard_index': 3}, 'depends_on': ['calibration-neutral', 'calibration-regions', 'calibration-rgb', 'freeze', 'setup-dino'], 'id': 'calibration-cache-03', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'expected_region_lock_sha256': {'$resource': 'calibration-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'calibration-cache-04-features'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'calibration-neutral'}, 'output_root': {'$output': 'calibration-cache-04'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'calibration-regions'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'role': 'calibration', 'shard_count': 6, 'shard_index': 4}, 'depends_on': ['calibration-neutral', 'calibration-regions', 'calibration-rgb', 'freeze', 'setup-dino'], 'id': 'calibration-cache-04', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'expected_region_lock_sha256': {'$resource': 'calibration-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'calibration-cache-05-features'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'calibration-neutral'}, 'output_root': {'$output': 'calibration-cache-05'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'calibration-regions'}, 'rgb_root': {'$resource': 'calibration-rgb'}, 'role': 'calibration', 'shard_count': 6, 'shard_index': 5}, 'depends_on': ['calibration-neutral', 'calibration-regions', 'calibration-rgb', 'freeze', 'setup-dino'], 'id': 'calibration-cache-05', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'config': {'action': 'cache_merge', 'cache_shards': [{'$resource': 'calibration-cache-00'}, {'$resource': 'calibration-cache-01'}, {'$resource': 'calibration-cache-02'}, {'$resource': 'calibration-cache-03'}, {'$resource': 'calibration-cache-04'}, {'$resource': 'calibration-cache-05'}], 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'maximum_output_bytes': 19327352832, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': 'calibration-cache'}, 'role': 'calibration'}, 'depends_on': ['calibration-cache-00', 'calibration-cache-01', 'calibration-cache-02', 'calibration-cache-03', 'calibration-cache-04', 'calibration-cache-05', 'freeze', 'setup-cpu'], 'id': 'calibration-cache', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'cache merge', 'runtime': 'cpu'}, {'config': {'action': 'release', 'cache_root': {'$resource': 'calibration-cache'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': '08f1'}, 'role': 'calibration'}, 'depends_on': ['calibration-cache', 'freeze', 'setup-cpu'], 'id': '08f1', 'input_boundary': 'model_only', 'notebook': 'notebooks/protected-evaluation/08f1-toysad8k-calibration-resource-release.ipynb', 'purpose': 'release', 'runtime': 'cpu'}, {'config': {'action': 'predict', 'cache_root': {'$resource': '08f1'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': '09a'}, 'role': 'calibration'}, 'depends_on': ['08f1', 'freeze', 'setup-cpu'], 'id': '09a', 'input_boundary': 'model_only', 'notebook': 'notebooks/protected-evaluation/09a-toysad8k-protected-calibration-inference.ipynb', 'purpose': 'predict', 'runtime': 'cpu'}, {'config': {'action': 'raw_truth', 'administrative_root': {'$resource': 'administrative'}, 'cache_root': {'$resource': '08f1'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'geometry_root': {'$resource': 'calibration-geometry'}, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': 'calibration-truth'}, 'prediction_root': {'$resource': '09a'}, 'raw_root': {'$resource': 'raw'}, 'rgb_index_root': {'$resource': 'calibration-index'}, 'role': 'calibration'}, 'depends_on': ['08f1', '09a', 'calibration-geometry', 'calibration-index', 'freeze', 'setup-cpu'], 'id': 'calibration-truth', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'raw truth', 'runtime': 'cpu'}, {'attach_resources': ['runtime-tracking', 'wheels-tracking'], 'config': {'action': 'evaluate', 'cache_root': {'$resource': '08f1'}, 'evaluator_root': {'$resource': 'calibration-truth'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': '09b'}, 'prediction_root': {'$resource': '09a'}, 'publish_tracking': True, 'report_root': {'$output': '09b-report'}, 'role': 'calibration'}, 'depends_on': ['08f1', '09a', 'calibration-truth', 'freeze', 'setup-cpu', 'setup-tracking'], 'id': '09b', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/protected-evaluation/09b-toysad8k-calibration-review-policy.ipynb', 'purpose': 'evaluate', 'runtime': 'cpu'}, {'config': {'action': 'rgb_release', 'calibration_receipt_root': {'$resource': '09b'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'maximum_new_scenes': 400, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': 'internal_lock-rgb'}, 'raw_root': {'$resource': 'raw'}, 'rgb_index_root': {'$resource': 'internal_lock-index'}, 'role': 'internal_lock'}, 'depends_on': ['09b', 'freeze', 'internal_lock-geometry', 'internal_lock-index', 'setup-cpu'], 'id': 'internal_lock-rgb', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'rgb release', 'runtime': 'cpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'calibration_receipt_root': {'$resource': '09b'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': 'internal_lock-regions-00'}, 'rgb_root': {'$resource': 'internal_lock-rgb'}, 'role': 'internal_lock', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 4, 'shard_index': 0}, 'depends_on': ['09b', 'freeze', 'internal_lock-rgb', 'setup-dino', 'setup-regions'], 'id': 'internal_lock-regions-00', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'calibration_receipt_root': {'$resource': '09b'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': 'internal_lock-regions-01'}, 'rgb_root': {'$resource': 'internal_lock-rgb'}, 'role': 'internal_lock', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 4, 'shard_index': 1}, 'depends_on': ['09b', 'freeze', 'internal_lock-rgb', 'setup-dino', 'setup-regions'], 'id': 'internal_lock-regions-01', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'calibration_receipt_root': {'$resource': '09b'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': 'internal_lock-regions-02'}, 'rgb_root': {'$resource': 'internal_lock-rgb'}, 'role': 'internal_lock', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 4, 'shard_index': 2}, 'depends_on': ['09b', 'freeze', 'internal_lock-rgb', 'setup-dino', 'setup-regions'], 'id': 'internal_lock-regions-02', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'calibration_receipt_root': {'$resource': '09b'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': 'internal_lock-regions-03'}, 'rgb_root': {'$resource': 'internal_lock-rgb'}, 'role': 'internal_lock', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 4, 'shard_index': 3}, 'depends_on': ['09b', 'freeze', 'internal_lock-rgb', 'setup-dino', 'setup-regions'], 'id': 'internal_lock-regions-03', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'region_merge', 'calibration_receipt_root': {'$resource': '09b'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': 'internal_lock-regions'}, 'region_shards': [{'$resource': 'internal_lock-regions-00'}, {'$resource': 'internal_lock-regions-01'}, {'$resource': 'internal_lock-regions-02'}, {'$resource': 'internal_lock-regions-03'}], 'rgb_root': {'$resource': 'internal_lock-rgb'}, 'role': 'internal_lock'}, 'depends_on': ['09b', 'freeze', 'internal_lock-regions-00', 'internal_lock-regions-01', 'internal_lock-regions-02', 'internal_lock-regions-03', 'internal_lock-rgb', 'setup-cpu'], 'id': 'internal_lock-regions', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'region merge', 'runtime': 'cpu'}, {'config': {'action': 'neutral', 'calibration_receipt_root': {'$resource': '09b'}, 'expected_region_lock_sha256': {'$resource': 'internal_lock-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'geometry_root': {'$resource': 'internal_lock-geometry'}, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': 'internal_lock-neutral'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'internal_lock-regions'}, 'rgb_root': {'$resource': 'internal_lock-rgb'}, 'role': 'internal_lock'}, 'depends_on': ['09b', 'freeze', 'internal_lock-geometry', 'internal_lock-regions', 'internal_lock-rgb', 'setup-cpu'], 'id': 'internal_lock-neutral', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'neutral', 'runtime': 'cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'calibration_receipt_root': {'$resource': '09b'}, 'expected_region_lock_sha256': {'$resource': 'internal_lock-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'internal_lock-cache-00-features'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'internal_lock-neutral'}, 'output_root': {'$output': 'internal_lock-cache-00'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'internal_lock-regions'}, 'rgb_root': {'$resource': 'internal_lock-rgb'}, 'role': 'internal_lock', 'shard_count': 4, 'shard_index': 0}, 'depends_on': ['09b', 'freeze', 'internal_lock-neutral', 'internal_lock-regions', 'internal_lock-rgb', 'setup-dino'], 'id': 'internal_lock-cache-00', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'calibration_receipt_root': {'$resource': '09b'}, 'expected_region_lock_sha256': {'$resource': 'internal_lock-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'internal_lock-cache-01-features'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'internal_lock-neutral'}, 'output_root': {'$output': 'internal_lock-cache-01'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'internal_lock-regions'}, 'rgb_root': {'$resource': 'internal_lock-rgb'}, 'role': 'internal_lock', 'shard_count': 4, 'shard_index': 1}, 'depends_on': ['09b', 'freeze', 'internal_lock-neutral', 'internal_lock-regions', 'internal_lock-rgb', 'setup-dino'], 'id': 'internal_lock-cache-01', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'calibration_receipt_root': {'$resource': '09b'}, 'expected_region_lock_sha256': {'$resource': 'internal_lock-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'internal_lock-cache-02-features'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'internal_lock-neutral'}, 'output_root': {'$output': 'internal_lock-cache-02'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'internal_lock-regions'}, 'rgb_root': {'$resource': 'internal_lock-rgb'}, 'role': 'internal_lock', 'shard_count': 4, 'shard_index': 2}, 'depends_on': ['09b', 'freeze', 'internal_lock-neutral', 'internal_lock-regions', 'internal_lock-rgb', 'setup-dino'], 'id': 'internal_lock-cache-02', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'calibration_receipt_root': {'$resource': '09b'}, 'expected_region_lock_sha256': {'$resource': 'internal_lock-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'internal_lock-cache-03-features'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'internal_lock-neutral'}, 'output_root': {'$output': 'internal_lock-cache-03'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'internal_lock-regions'}, 'rgb_root': {'$resource': 'internal_lock-rgb'}, 'role': 'internal_lock', 'shard_count': 4, 'shard_index': 3}, 'depends_on': ['09b', 'freeze', 'internal_lock-neutral', 'internal_lock-regions', 'internal_lock-rgb', 'setup-dino'], 'id': 'internal_lock-cache-03', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'config': {'action': 'cache_merge', 'cache_shards': [{'$resource': 'internal_lock-cache-00'}, {'$resource': 'internal_lock-cache-01'}, {'$resource': 'internal_lock-cache-02'}, {'$resource': 'internal_lock-cache-03'}], 'calibration_receipt_root': {'$resource': '09b'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'maximum_output_bytes': 19327352832, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': 'internal_lock-cache'}, 'role': 'internal_lock'}, 'depends_on': ['09b', 'freeze', 'internal_lock-cache-00', 'internal_lock-cache-01', 'internal_lock-cache-02', 'internal_lock-cache-03', 'setup-cpu'], 'id': 'internal_lock-cache', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'cache merge', 'runtime': 'cpu'}, {'config': {'action': 'release', 'cache_root': {'$resource': 'internal_lock-cache'}, 'calibration_receipt_root': {'$resource': '09b'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': '09b1'}, 'role': 'internal_lock'}, 'depends_on': ['09b', 'freeze', 'internal_lock-cache', 'setup-cpu'], 'id': '09b1', 'input_boundary': 'model_only', 'notebook': 'notebooks/protected-evaluation/09b1-toysad8k-internal-lock-resource-release.ipynb', 'purpose': 'release', 'runtime': 'cpu'}, {'config': {'action': 'predict', 'cache_root': {'$resource': '09b1'}, 'calibration_receipt_root': {'$resource': '09b'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': '09c'}, 'role': 'internal_lock'}, 'depends_on': ['09b', '09b1', 'freeze', 'setup-cpu'], 'id': '09c', 'input_boundary': 'model_only', 'notebook': 'notebooks/protected-evaluation/09c-toysad8k-protected-internal-lock-inference.ipynb', 'purpose': 'predict', 'runtime': 'cpu'}, {'config': {'action': 'raw_truth', 'administrative_root': {'$resource': 'administrative'}, 'cache_root': {'$resource': '09b1'}, 'calibration_receipt_root': {'$resource': '09b'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'geometry_root': {'$resource': 'internal_lock-geometry'}, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': 'internal_lock-truth'}, 'prediction_root': {'$resource': '09c'}, 'raw_root': {'$resource': 'raw'}, 'rgb_index_root': {'$resource': 'internal_lock-index'}, 'role': 'internal_lock'}, 'depends_on': ['09b', '09b1', '09c', 'freeze', 'internal_lock-geometry', 'internal_lock-index', 'setup-cpu'], 'id': 'internal_lock-truth', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'raw truth', 'runtime': 'cpu'}, {'attach_resources': ['runtime-tracking', 'wheels-tracking'], 'config': {'action': 'evaluate', 'cache_root': {'$resource': '09b1'}, 'calibration_receipt_root': {'$resource': '09b'}, 'evaluator_root': {'$resource': 'internal_lock-truth'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'method_freeze_root': {'$resource': 'freeze'}, 'output_root': {'$output': '09d'}, 'prediction_root': {'$resource': '09c'}, 'publish_tracking': True, 'report_root': {'$output': '09d-report'}, 'role': 'internal_lock'}, 'depends_on': ['09b', '09b1', '09c', 'freeze', 'internal_lock-truth', 'setup-cpu', 'setup-tracking'], 'id': '09d', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/protected-evaluation/09d-toysad8k-internal-lock-one-shot-evaluation.ipynb', 'purpose': 'evaluate', 'runtime': 'cpu'}, {'config': {'action': 'rgb_release', 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'maximum_new_scenes': 715, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'seen-rgb'}, 'raw_root': {'$resource': 'raw'}, 'rgb_index_root': {'$resource': 'seen-index'}, 'role': 'seen'}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-geometry', 'seen-index', 'setup-cpu'], 'id': 'seen-rgb', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'rgb release', 'runtime': 'cpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'seen-regions-00'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'role': 'seen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 8, 'shard_index': 0}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-rgb', 'setup-dino', 'setup-regions'], 'id': 'seen-regions-00', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'seen-regions-01'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'role': 'seen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 8, 'shard_index': 1}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-rgb', 'setup-dino', 'setup-regions'], 'id': 'seen-regions-01', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'seen-regions-02'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'role': 'seen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 8, 'shard_index': 2}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-rgb', 'setup-dino', 'setup-regions'], 'id': 'seen-regions-02', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'seen-regions-03'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'role': 'seen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 8, 'shard_index': 3}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-rgb', 'setup-dino', 'setup-regions'], 'id': 'seen-regions-03', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'seen-regions-04'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'role': 'seen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 8, 'shard_index': 4}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-rgb', 'setup-dino', 'setup-regions'], 'id': 'seen-regions-04', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'seen-regions-05'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'role': 'seen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 8, 'shard_index': 5}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-rgb', 'setup-dino', 'setup-regions'], 'id': 'seen-regions-05', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'seen-regions-06'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'role': 'seen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 8, 'shard_index': 6}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-rgb', 'setup-dino', 'setup-regions'], 'id': 'seen-regions-06', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'seen-regions-07'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'role': 'seen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 8, 'shard_index': 7}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-rgb', 'setup-dino', 'setup-regions'], 'id': 'seen-regions-07', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'region_merge', 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'seen-regions'}, 'region_shards': [{'$resource': 'seen-regions-00'}, {'$resource': 'seen-regions-01'}, {'$resource': 'seen-regions-02'}, {'$resource': 'seen-regions-03'}, {'$resource': 'seen-regions-04'}, {'$resource': 'seen-regions-05'}, {'$resource': 'seen-regions-06'}, {'$resource': 'seen-regions-07'}], 'rgb_root': {'$resource': 'seen-rgb'}, 'role': 'seen'}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-regions-00', 'seen-regions-01', 'seen-regions-02', 'seen-regions-03', 'seen-regions-04', 'seen-regions-05', 'seen-regions-06', 'seen-regions-07', 'seen-rgb', 'setup-cpu'], 'id': 'seen-regions', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'region merge', 'runtime': 'cpu'}, {'config': {'action': 'neutral', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'seen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'geometry_root': {'$resource': 'seen-geometry'}, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'seen-neutral'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'seen-regions'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'role': 'seen'}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-geometry', 'seen-regions', 'seen-rgb', 'setup-cpu'], 'id': 'seen-neutral', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'neutral', 'runtime': 'cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'seen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'seen-cache-00-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'seen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'seen-cache-00'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'seen-regions'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'role': 'seen', 'shard_count': 8, 'shard_index': 0}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-neutral', 'seen-regions', 'seen-rgb', 'setup-dino'], 'id': 'seen-cache-00', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'seen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'seen-cache-01-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'seen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'seen-cache-01'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'seen-regions'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'role': 'seen', 'shard_count': 8, 'shard_index': 1}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-neutral', 'seen-regions', 'seen-rgb', 'setup-dino'], 'id': 'seen-cache-01', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'seen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'seen-cache-02-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'seen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'seen-cache-02'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'seen-regions'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'role': 'seen', 'shard_count': 8, 'shard_index': 2}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-neutral', 'seen-regions', 'seen-rgb', 'setup-dino'], 'id': 'seen-cache-02', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'seen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'seen-cache-03-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'seen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'seen-cache-03'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'seen-regions'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'role': 'seen', 'shard_count': 8, 'shard_index': 3}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-neutral', 'seen-regions', 'seen-rgb', 'setup-dino'], 'id': 'seen-cache-03', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'seen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'seen-cache-04-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'seen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'seen-cache-04'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'seen-regions'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'role': 'seen', 'shard_count': 8, 'shard_index': 4}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-neutral', 'seen-regions', 'seen-rgb', 'setup-dino'], 'id': 'seen-cache-04', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'seen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'seen-cache-05-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'seen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'seen-cache-05'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'seen-regions'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'role': 'seen', 'shard_count': 8, 'shard_index': 5}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-neutral', 'seen-regions', 'seen-rgb', 'setup-dino'], 'id': 'seen-cache-05', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'seen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'seen-cache-06-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'seen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'seen-cache-06'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'seen-regions'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'role': 'seen', 'shard_count': 8, 'shard_index': 6}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-neutral', 'seen-regions', 'seen-rgb', 'setup-dino'], 'id': 'seen-cache-06', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'seen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'seen-cache-07-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'seen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'seen-cache-07'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'seen-regions'}, 'rgb_root': {'$resource': 'seen-rgb'}, 'role': 'seen', 'shard_count': 8, 'shard_index': 7}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-neutral', 'seen-regions', 'seen-rgb', 'setup-dino'], 'id': 'seen-cache-07', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'config': {'action': 'cache_merge', 'cache_shards': [{'$resource': 'seen-cache-00'}, {'$resource': 'seen-cache-01'}, {'$resource': 'seen-cache-02'}, {'$resource': 'seen-cache-03'}, {'$resource': 'seen-cache-04'}, {'$resource': 'seen-cache-05'}, {'$resource': 'seen-cache-06'}, {'$resource': 'seen-cache-07'}], 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'maximum_output_bytes': 19327352832, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'seen-cache'}, 'role': 'seen'}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-cache-00', 'seen-cache-01', 'seen-cache-02', 'seen-cache-03', 'seen-cache-04', 'seen-cache-05', 'seen-cache-06', 'seen-cache-07', 'setup-cpu'], 'id': 'seen-cache', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'cache merge', 'runtime': 'cpu'}, {'config': {'action': 'release', 'cache_root': {'$resource': 'seen-cache'}, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': '10r-seen'}, 'role': 'seen'}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'seen-cache', 'setup-cpu'], 'id': '10r-seen', 'input_boundary': 'model_only', 'notebook': 'notebooks/official-evaluation/10r-toysad8k-official-release.ipynb', 'purpose': 'release', 'runtime': 'cpu'}, {'config': {'action': 'predict', 'cache_root': {'$resource': '10r-seen'}, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'method_freeze_root': {'$resource': 'freeze'}, 'mode': 'automatic', 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': '10a-automatic'}, 'role': 'seen'}, 'depends_on': ['09d', '10r-seen', 'freeze', 'official-exposure', 'setup-cpu'], 'id': '10a-automatic', 'input_boundary': 'model_only', 'notebook': 'notebooks/official-evaluation/10a-toysad8k-official-seen-frozen-inference.ipynb', 'purpose': 'predict', 'runtime': 'cpu'}, {'config': {'action': 'predict', 'cache_root': {'$resource': '10r-seen'}, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'method_freeze_root': {'$resource': 'freeze'}, 'mode': 'assisted', 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': '10a-assisted'}, 'role': 'seen'}, 'depends_on': ['09d', '10r-seen', 'freeze', 'official-exposure', 'setup-cpu'], 'id': '10a-assisted', 'input_boundary': 'model_only', 'notebook': 'notebooks/official-evaluation/10a-toysad8k-official-seen-frozen-inference.ipynb', 'purpose': 'predict', 'runtime': 'cpu'}, {'config': {'action': 'raw_truth', 'cache_root': {'$resource': '10r-seen'}, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'geometry_root': {'$resource': 'seen-geometry'}, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'seen-truth'}, 'prediction_roots': {'assisted': {'$resource': '10a-assisted'}, 'automatic': {'$resource': '10a-automatic'}}, 'raw_root': {'$resource': 'raw'}, 'rgb_index_root': {'$resource': 'seen-index'}, 'role': 'seen'}, 'depends_on': ['09d', '10a-assisted', '10a-automatic', '10r-seen', 'freeze', 'official-exposure', 'seen-geometry', 'seen-index', 'setup-cpu'], 'id': 'seen-truth', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'raw truth', 'runtime': 'cpu'}, {'config': {'action': 'rgb_release', 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'maximum_new_scenes': 2437, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-rgb'}, 'raw_root': {'$resource': 'raw'}, 'rgb_index_root': {'$resource': 'unseen-index'}, 'role': 'unseen'}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-cpu', 'unseen-geometry', 'unseen-index'], 'id': 'unseen-rgb', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'rgb release', 'runtime': 'cpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-regions-00'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 0}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'id': 'unseen-regions-00', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-regions-01'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 1}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'id': 'unseen-regions-01', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-regions-02'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 2}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'id': 'unseen-regions-02', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-regions-03'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 3}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'id': 'unseen-regions-03', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-regions-04'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 4}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'id': 'unseen-regions-04', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-regions-05'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 5}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'id': 'unseen-regions-05', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-regions-06'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 6}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'id': 'unseen-regions-06', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-regions-07'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 7}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'id': 'unseen-regions-07', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-regions-08'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 8}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'id': 'unseen-regions-08', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-regions-09'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 9}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'id': 'unseen-regions-09', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-regions-10'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 10}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'id': 'unseen-regions-10', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-regions-11'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 11}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'id': 'unseen-regions-11', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-regions-12'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 12}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'id': 'unseen-regions-12', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-regions-13'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 13}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'id': 'unseen-regions-13', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-regions-14'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 14}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'id': 'unseen-regions-14', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-regions-15'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 15}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'id': 'unseen-regions-15', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-regions-16'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 16}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'id': 'unseen-regions-16', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-regions-17'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 17}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'id': 'unseen-regions-17', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-regions-18'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 18}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'id': 'unseen-regions-18', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-regions-19'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 19}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'id': 'unseen-regions-19', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-regions-20'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 20}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'id': 'unseen-regions-20', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-regions-21'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 21}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'id': 'unseen-regions-21', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-regions-22'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 22}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'id': 'unseen-regions-22', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-regions-23'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 23}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'id': 'unseen-regions-23', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'extract', 'attempt_timeout_seconds': 600, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'locate_model_root': {'$resource': 'models', 'path': 'locateanything'}, 'maximum_new_views': 2000, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-regions-24'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'sam_checkpoint': {'$resource': 'models', 'path': 'sam2/sam2.1_hiera_large.pt'}, 'shard_count': 25, 'shard_index': 24}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'setup-regions', 'unseen-rgb'], 'id': 'unseen-regions-24', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-region-inference.ipynb', 'purpose': 'Fixed LocateAnything/SAM regions for at most100 scenes; checkpoint each phase', 'runtime': 'regions-gpu'}, {'config': {'action': 'region_merge', 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-regions'}, 'region_shards': [{'$resource': 'unseen-regions-00'}, {'$resource': 'unseen-regions-01'}, {'$resource': 'unseen-regions-02'}, {'$resource': 'unseen-regions-03'}, {'$resource': 'unseen-regions-04'}, {'$resource': 'unseen-regions-05'}, {'$resource': 'unseen-regions-06'}, {'$resource': 'unseen-regions-07'}, {'$resource': 'unseen-regions-08'}, {'$resource': 'unseen-regions-09'}, {'$resource': 'unseen-regions-10'}, {'$resource': 'unseen-regions-11'}, {'$resource': 'unseen-regions-12'}, {'$resource': 'unseen-regions-13'}, {'$resource': 'unseen-regions-14'}, {'$resource': 'unseen-regions-15'}, {'$resource': 'unseen-regions-16'}, {'$resource': 'unseen-regions-17'}, {'$resource': 'unseen-regions-18'}, {'$resource': 'unseen-regions-19'}, {'$resource': 'unseen-regions-20'}, {'$resource': 'unseen-regions-21'}, {'$resource': 'unseen-regions-22'}, {'$resource': 'unseen-regions-23'}, {'$resource': 'unseen-regions-24'}], 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen'}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-cpu', 'unseen-regions-00', 'unseen-regions-01', 'unseen-regions-02', 'unseen-regions-03', 'unseen-regions-04', 'unseen-regions-05', 'unseen-regions-06', 'unseen-regions-07', 'unseen-regions-08', 'unseen-regions-09', 'unseen-regions-10', 'unseen-regions-11', 'unseen-regions-12', 'unseen-regions-13', 'unseen-regions-14', 'unseen-regions-15', 'unseen-regions-16', 'unseen-regions-17', 'unseen-regions-18', 'unseen-regions-19', 'unseen-regions-20', 'unseen-regions-21', 'unseen-regions-22', 'unseen-regions-23', 'unseen-regions-24', 'unseen-rgb'], 'id': 'unseen-regions', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'region merge', 'runtime': 'cpu'}, {'config': {'action': 'neutral', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'geometry_root': {'$resource': 'unseen-geometry'}, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-neutral'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen'}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-cpu', 'unseen-geometry', 'unseen-regions', 'unseen-rgb'], 'id': 'unseen-neutral', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'neutral', 'runtime': 'cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'unseen-cache-00-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-cache-00'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'shard_count': 25, 'shard_index': 0}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'id': 'unseen-cache-00', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'unseen-cache-01-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-cache-01'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'shard_count': 25, 'shard_index': 1}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'id': 'unseen-cache-01', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'unseen-cache-02-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-cache-02'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'shard_count': 25, 'shard_index': 2}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'id': 'unseen-cache-02', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'unseen-cache-03-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-cache-03'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'shard_count': 25, 'shard_index': 3}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'id': 'unseen-cache-03', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'unseen-cache-04-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-cache-04'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'shard_count': 25, 'shard_index': 4}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'id': 'unseen-cache-04', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'unseen-cache-05-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-cache-05'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'shard_count': 25, 'shard_index': 5}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'id': 'unseen-cache-05', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'unseen-cache-06-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-cache-06'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'shard_count': 25, 'shard_index': 6}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'id': 'unseen-cache-06', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'unseen-cache-07-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-cache-07'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'shard_count': 25, 'shard_index': 7}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'id': 'unseen-cache-07', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'unseen-cache-08-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-cache-08'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'shard_count': 25, 'shard_index': 8}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'id': 'unseen-cache-08', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'unseen-cache-09-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-cache-09'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'shard_count': 25, 'shard_index': 9}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'id': 'unseen-cache-09', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'unseen-cache-10-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-cache-10'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'shard_count': 25, 'shard_index': 10}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'id': 'unseen-cache-10', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'unseen-cache-11-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-cache-11'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'shard_count': 25, 'shard_index': 11}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'id': 'unseen-cache-11', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'unseen-cache-12-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-cache-12'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'shard_count': 25, 'shard_index': 12}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'id': 'unseen-cache-12', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'unseen-cache-13-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-cache-13'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'shard_count': 25, 'shard_index': 13}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'id': 'unseen-cache-13', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'unseen-cache-14-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-cache-14'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'shard_count': 25, 'shard_index': 14}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'id': 'unseen-cache-14', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'unseen-cache-15-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-cache-15'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'shard_count': 25, 'shard_index': 15}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'id': 'unseen-cache-15', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'unseen-cache-16-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-cache-16'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'shard_count': 25, 'shard_index': 16}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'id': 'unseen-cache-16', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'unseen-cache-17-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-cache-17'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'shard_count': 25, 'shard_index': 17}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'id': 'unseen-cache-17', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'unseen-cache-18-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-cache-18'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'shard_count': 25, 'shard_index': 18}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'id': 'unseen-cache-18', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'unseen-cache-19-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-cache-19'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'shard_count': 25, 'shard_index': 19}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'id': 'unseen-cache-19', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'unseen-cache-20-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-cache-20'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'shard_count': 25, 'shard_index': 20}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'id': 'unseen-cache-20', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'unseen-cache-21-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-cache-21'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'shard_count': 25, 'shard_index': 21}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'id': 'unseen-cache-21', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'unseen-cache-22-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-cache-22'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'shard_count': 25, 'shard_index': 22}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'id': 'unseen-cache-22', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'unseen-cache-23-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-cache-23'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'shard_count': 25, 'shard_index': 23}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'id': 'unseen-cache-23', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'compact_checkpoints': True, 'config': {'action': 'prepare_shard', 'decision_root': {'$resource': '09d'}, 'expected_region_lock_sha256': {'$resource': 'unseen-regions', 'path': 'artifact_lock.json', 'sha256': True}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'feature_root': {'$output': 'unseen-cache-24-features'}, 'freeze_root': {'$resource': 'freeze'}, 'max_new_scenes': 100, 'max_new_units': 2000, 'maximum_output_bytes': 16106127360, 'method_freeze_root': {'$resource': 'freeze'}, 'model_source': {'$resource': 'models', 'path': 'dino'}, 'neutral_root': {'$resource': 'unseen-neutral'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-cache-24'}, 'region_format': 'current-native-regions', 'region_root': {'$resource': 'unseen-regions'}, 'rgb_root': {'$resource': 'unseen-rgb'}, 'role': 'unseen', 'shard_count': 25, 'shard_index': 24}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-dino', 'unseen-neutral', 'unseen-regions', 'unseen-rgb'], 'id': 'unseen-cache-24', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb', 'purpose': 'Reuse compatible features and extract only missing fixed DINO crops; assemble at most100 scenes', 'runtime': 'dino-cpu'}, {'config': {'action': 'cache_merge', 'cache_shards': [{'$resource': 'unseen-cache-00'}, {'$resource': 'unseen-cache-01'}, {'$resource': 'unseen-cache-02'}, {'$resource': 'unseen-cache-03'}, {'$resource': 'unseen-cache-04'}, {'$resource': 'unseen-cache-05'}, {'$resource': 'unseen-cache-06'}, {'$resource': 'unseen-cache-07'}, {'$resource': 'unseen-cache-08'}, {'$resource': 'unseen-cache-09'}, {'$resource': 'unseen-cache-10'}, {'$resource': 'unseen-cache-11'}, {'$resource': 'unseen-cache-12'}, {'$resource': 'unseen-cache-13'}, {'$resource': 'unseen-cache-14'}, {'$resource': 'unseen-cache-15'}, {'$resource': 'unseen-cache-16'}, {'$resource': 'unseen-cache-17'}, {'$resource': 'unseen-cache-18'}, {'$resource': 'unseen-cache-19'}, {'$resource': 'unseen-cache-20'}, {'$resource': 'unseen-cache-21'}, {'$resource': 'unseen-cache-22'}, {'$resource': 'unseen-cache-23'}, {'$resource': 'unseen-cache-24'}], 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'maximum_output_bytes': 19327352832, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-cache'}, 'role': 'unseen'}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-cpu', 'unseen-cache-00', 'unseen-cache-01', 'unseen-cache-02', 'unseen-cache-03', 'unseen-cache-04', 'unseen-cache-05', 'unseen-cache-06', 'unseen-cache-07', 'unseen-cache-08', 'unseen-cache-09', 'unseen-cache-10', 'unseen-cache-11', 'unseen-cache-12', 'unseen-cache-13', 'unseen-cache-14', 'unseen-cache-15', 'unseen-cache-16', 'unseen-cache-17', 'unseen-cache-18', 'unseen-cache-19', 'unseen-cache-20', 'unseen-cache-21', 'unseen-cache-22', 'unseen-cache-23', 'unseen-cache-24'], 'id': 'unseen-cache', 'input_boundary': 'model_only', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'cache merge', 'runtime': 'cpu'}, {'config': {'action': 'release', 'cache_root': {'$resource': 'unseen-cache'}, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': '10r-unseen'}, 'role': 'unseen'}, 'depends_on': ['09d', 'freeze', 'official-exposure', 'setup-cpu', 'unseen-cache'], 'id': '10r-unseen', 'input_boundary': 'model_only', 'notebook': 'notebooks/official-evaluation/10r-toysad8k-official-release.ipynb', 'purpose': 'release', 'runtime': 'cpu'}, {'config': {'action': 'predict', 'cache_root': {'$resource': '10r-unseen'}, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'method_freeze_root': {'$resource': 'freeze'}, 'mode': 'automatic', 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': '10a1-automatic'}, 'role': 'unseen'}, 'depends_on': ['09d', '10r-unseen', 'freeze', 'official-exposure', 'setup-cpu'], 'id': '10a1-automatic', 'input_boundary': 'model_only', 'notebook': 'notebooks/official-evaluation/10a1-toysad8k-official-unseen-frozen-inference.ipynb', 'purpose': 'predict', 'runtime': 'cpu'}, {'config': {'action': 'predict', 'cache_root': {'$resource': '10r-unseen'}, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'method_freeze_root': {'$resource': 'freeze'}, 'mode': 'assisted', 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': '10a1-assisted'}, 'role': 'unseen'}, 'depends_on': ['09d', '10r-unseen', 'freeze', 'official-exposure', 'setup-cpu'], 'id': '10a1-assisted', 'input_boundary': 'model_only', 'notebook': 'notebooks/official-evaluation/10a1-toysad8k-official-unseen-frozen-inference.ipynb', 'purpose': 'predict', 'runtime': 'cpu'}, {'config': {'action': 'raw_truth', 'cache_root': {'$resource': '10r-unseen'}, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'geometry_root': {'$resource': 'unseen-geometry'}, 'method_freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': 'unseen-truth'}, 'prediction_roots': {'assisted': {'$resource': '10a1-assisted'}, 'automatic': {'$resource': '10a1-automatic'}}, 'raw_root': {'$resource': 'raw'}, 'rgb_index_root': {'$resource': 'unseen-index'}, 'role': 'unseen'}, 'depends_on': ['09d', '10a1-assisted', '10a1-automatic', '10r-unseen', 'freeze', 'official-exposure', 'setup-cpu', 'unseen-geometry', 'unseen-index'], 'id': 'unseen-truth', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/input-preparation/current-kaggle-preparation.ipynb', 'purpose': 'raw truth', 'runtime': 'cpu'}, {'config': {'action': 'finalize', 'cache_roots': {'seen': {'$resource': '10r-seen'}, 'unseen': {'$resource': '10r-unseen'}}, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'mode': 'automatic', 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': '10a-finalize-automatic'}, 'prediction_roots': {'seen': {'$resource': '10a-automatic'}, 'unseen': {'$resource': '10a1-automatic'}}}, 'depends_on': ['09d', '10a-automatic', '10a1-automatic', '10r-seen', '10r-unseen', 'freeze', 'official-exposure', 'setup-cpu'], 'id': '10a-finalize-automatic', 'input_boundary': 'model_only', 'notebook': 'notebooks/official-evaluation/10a-finalize-toysad8k-official-prediction-summary.ipynb', 'purpose': 'finalize', 'runtime': 'cpu'}, {'config': {'action': 'finalize', 'cache_roots': {'seen': {'$resource': '10r-seen'}, 'unseen': {'$resource': '10r-unseen'}}, 'decision_root': {'$resource': '09d'}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'mode': 'assisted', 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': '10a-finalize-assisted'}, 'prediction_roots': {'seen': {'$resource': '10a-assisted'}, 'unseen': {'$resource': '10a1-assisted'}}}, 'depends_on': ['09d', '10a-assisted', '10a1-assisted', '10r-seen', '10r-unseen', 'freeze', 'official-exposure', 'setup-cpu'], 'id': '10a-finalize-assisted', 'input_boundary': 'model_only', 'notebook': 'notebooks/official-evaluation/10a-finalize-toysad8k-official-prediction-summary.ipynb', 'purpose': 'finalize', 'runtime': 'cpu'}, {'attach_resources': ['runtime-tracking', 'wheels-tracking'], 'config': {'action': 'evaluate', 'cache_roots': {'seen': {'$resource': '10r-seen'}, 'unseen': {'$resource': '10r-unseen'}}, 'decision_root': {'$resource': '09d'}, 'evaluator_roots': {'seen': {'$resource': 'seen-truth'}, 'unseen': {'$resource': 'unseen-truth'}}, 'exposure_root': {'$resource': 'support', 'path': 'original-exposure'}, 'freeze_root': {'$resource': 'freeze'}, 'official_exposure_root': {'$resource': 'official-exposure'}, 'output_root': {'$output': '10b'}, 'prediction_roots': {'assisted': {'seen': {'$resource': '10a-assisted'}, 'unseen': {'$resource': '10a1-assisted'}}, 'automatic': {'seen': {'$resource': '10a-automatic'}, 'unseen': {'$resource': '10a1-automatic'}}}, 'prediction_summary_roots': {'assisted': {'$resource': '10a-finalize-assisted'}, 'automatic': {'$resource': '10a-finalize-automatic'}}, 'publish_tracking': True, 'report_root': {'$output': '10b-report'}}, 'depends_on': ['09d', '10a-assisted', '10a-automatic', '10a-finalize-assisted', '10a-finalize-automatic', '10a1-assisted', '10a1-automatic', '10r-seen', '10r-unseen', 'freeze', 'official-exposure', 'seen-truth', 'setup-cpu', 'setup-tracking', 'unseen-truth'], 'id': '10b', 'input_boundary': 'trusted_preparation', 'notebook': 'notebooks/official-evaluation/10b-toysad8k-official-seen-unseen-evaluation.ipynb', 'purpose': 'evaluate', 'runtime': 'cpu'}], 'methods': ['winner_cluster_consensus', 'object_bank_top2'], 'models_fixed': True, 'official_roles': {'seen': 715, 'unseen': 2437}, 'original_roles': {'calibration': 600, 'development': 1000, 'internal_lock': 400}, 'queue_note': 'Repeated cache jobs are100-scene resource shards of fixed methods, not new scoring experiments.', 'raw_dataset_reupload_required': False, 'resources': {'08f1': {'kind': 'kernel', 'marker': 'cohort_manifest.json', 'producer_job': '08f1', 'ref': 'esreekarreddy/vpi-current-08f1', 'suffixes': ['08f1/model-cache']}, '09a': {'kind': 'kernel', 'marker': 'prediction_lock.json', 'producer_job': '09a', 'ref': 'esreekarreddy/vpi-current-09a', 'suffixes': ['09a']}, '09b': {'contains_outcomes': True, 'kind': 'kernel', 'marker': 'evaluation_lock.json', 'producer_job': '09b', 'ref': 'esreekarreddy/vpi-current-09b', 'suffixes': ['09b']}, '09b1': {'kind': 'kernel', 'marker': 'cohort_manifest.json', 'producer_job': '09b1', 'ref': 'esreekarreddy/vpi-current-09b1', 'suffixes': ['09b1/model-cache']}, '09c': {'kind': 'kernel', 'marker': 'prediction_lock.json', 'producer_job': '09c', 'ref': 'esreekarreddy/vpi-current-09c', 'suffixes': ['09c']}, '09d': {'contains_outcomes': True, 'kind': 'kernel', 'marker': 'evaluation_lock.json', 'producer_job': '09d', 'ref': 'esreekarreddy/vpi-current-09d', 'suffixes': ['09d']}, '10a-assisted': {'kind': 'kernel', 'marker': 'prediction_lock.json', 'producer_job': '10a-assisted', 'ref': 'esreekarreddy/vpi-current-10a-assisted', 'suffixes': ['10a-assisted']}, '10a-automatic': {'kind': 'kernel', 'marker': 'prediction_lock.json', 'producer_job': '10a-automatic', 'ref': 'esreekarreddy/vpi-current-10a-automatic', 'suffixes': ['10a-automatic']}, '10a-finalize-assisted': {'kind': 'kernel', 'marker': 'summary_lock.json', 'producer_job': '10a-finalize-assisted', 'ref': 'esreekarreddy/vpi-current-10a-finalize-assisted', 'suffixes': ['10a-finalize-assisted']}, '10a-finalize-automatic': {'kind': 'kernel', 'marker': 'summary_lock.json', 'producer_job': '10a-finalize-automatic', 'ref': 'esreekarreddy/vpi-current-10a-finalize-automatic', 'suffixes': ['10a-finalize-automatic']}, '10a1-assisted': {'kind': 'kernel', 'marker': 'prediction_lock.json', 'producer_job': '10a1-assisted', 'ref': 'esreekarreddy/vpi-current-10a1-assisted', 'suffixes': ['10a1-assisted']}, '10a1-automatic': {'kind': 'kernel', 'marker': 'prediction_lock.json', 'producer_job': '10a1-automatic', 'ref': 'esreekarreddy/vpi-current-10a1-automatic', 'suffixes': ['10a1-automatic']}, '10b': {'contains_outcomes': True, 'kind': 'kernel', 'marker': 'evaluation_lock.json', 'producer_job': '10b', 'ref': 'esreekarreddy/vpi-current-10b', 'suffixes': ['10b']}, '10r-seen': {'kind': 'kernel', 'marker': 'cohort_manifest.json', 'producer_job': '10r-seen', 'ref': 'esreekarreddy/vpi-current-10r-seen', 'suffixes': ['10r-seen/model-input']}, '10r-unseen': {'kind': 'kernel', 'marker': 'cohort_manifest.json', 'producer_job': '10r-unseen', 'ref': 'esreekarreddy/vpi-current-10r-unseen', 'suffixes': ['10r-unseen/model-input']}, 'administrative': {'availability': 'existing_kaggle', 'kind': 'dataset', 'marker': 'restricted_source_map.jsonl', 'ref': 'esreekarreddy/vpi-toysad-selection-private', 'suffixes': ['toysad8k-train-2000-v2-administrative-vault']}, 'calibration-cache': {'kind': 'kernel', 'marker': 'cohort_manifest.json', 'producer_job': 'calibration-cache', 'ref': 'esreekarreddy/vpi-current-calibration-cache', 'suffixes': ['calibration-cache']}, 'calibration-cache-00': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'calibration-cache-00', 'ref': 'esreekarreddy/vpi-current-calibration-cache-00', 'suffixes': ['calibration-cache-00']}, 'calibration-cache-01': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'calibration-cache-01', 'ref': 'esreekarreddy/vpi-current-calibration-cache-01', 'suffixes': ['calibration-cache-01']}, 'calibration-cache-02': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'calibration-cache-02', 'ref': 'esreekarreddy/vpi-current-calibration-cache-02', 'suffixes': ['calibration-cache-02']}, 'calibration-cache-03': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'calibration-cache-03', 'ref': 'esreekarreddy/vpi-current-calibration-cache-03', 'suffixes': ['calibration-cache-03']}, 'calibration-cache-04': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'calibration-cache-04', 'ref': 'esreekarreddy/vpi-current-calibration-cache-04', 'suffixes': ['calibration-cache-04']}, 'calibration-cache-05': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'calibration-cache-05', 'ref': 'esreekarreddy/vpi-current-calibration-cache-05', 'suffixes': ['calibration-cache-05']}, 'calibration-geometry': {'kind': 'kernel', 'marker': 'geometry_manifest.json', 'producer_job': 'calibration-geometry', 'ref': 'esreekarreddy/vpi-current-calibration-geometry', 'suffixes': ['calibration-geometry']}, 'calibration-index': {'kind': 'kernel', 'marker': 'rgb_index.json', 'producer_job': 'calibration-index', 'ref': 'esreekarreddy/vpi-current-calibration-index', 'suffixes': ['calibration-index']}, 'calibration-neutral': {'kind': 'kernel', 'marker': 'neutral_correspondence.json', 'producer_job': 'calibration-neutral', 'ref': 'esreekarreddy/vpi-current-calibration-neutral', 'suffixes': ['calibration-neutral']}, 'calibration-regions': {'kind': 'kernel', 'marker': 'artifact_lock.json', 'producer_job': 'calibration-regions', 'ref': 'esreekarreddy/vpi-current-calibration-regions', 'suffixes': ['calibration-regions']}, 'calibration-regions-00': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'calibration-regions-00', 'ref': 'esreekarreddy/vpi-current-calibration-regions-00', 'suffixes': ['calibration-regions-00']}, 'calibration-regions-01': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'calibration-regions-01', 'ref': 'esreekarreddy/vpi-current-calibration-regions-01', 'suffixes': ['calibration-regions-01']}, 'calibration-regions-02': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'calibration-regions-02', 'ref': 'esreekarreddy/vpi-current-calibration-regions-02', 'suffixes': ['calibration-regions-02']}, 'calibration-regions-03': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'calibration-regions-03', 'ref': 'esreekarreddy/vpi-current-calibration-regions-03', 'suffixes': ['calibration-regions-03']}, 'calibration-regions-04': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'calibration-regions-04', 'ref': 'esreekarreddy/vpi-current-calibration-regions-04', 'suffixes': ['calibration-regions-04']}, 'calibration-regions-05': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'calibration-regions-05', 'ref': 'esreekarreddy/vpi-current-calibration-regions-05', 'suffixes': ['calibration-regions-05']}, 'calibration-rgb': {'kind': 'kernel', 'marker': 'package_record.json', 'producer_job': 'calibration-rgb', 'ref': 'esreekarreddy/vpi-current-calibration-rgb', 'suffixes': ['calibration-rgb/model-input']}, 'calibration-truth': {'contains_outcomes': True, 'kind': 'kernel', 'marker': 'current_evaluator_manifest.json', 'producer_job': 'calibration-truth', 'ref': 'esreekarreddy/vpi-current-calibration-truth', 'suffixes': ['calibration-truth']}, 'development-cache': {'kind': 'kernel', 'marker': 'cohort_manifest.json', 'producer_job': 'development-cache', 'ref': 'esreekarreddy/vpi-current-development-cache', 'suffixes': ['development-cache']}, 'development-cache-00': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'development-cache-00', 'ref': 'esreekarreddy/vpi-current-development-cache-00', 'suffixes': ['development-cache-00']}, 'development-cache-01': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'development-cache-01', 'ref': 'esreekarreddy/vpi-current-development-cache-01', 'suffixes': ['development-cache-01']}, 'development-cache-02': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'development-cache-02', 'ref': 'esreekarreddy/vpi-current-development-cache-02', 'suffixes': ['development-cache-02']}, 'development-cache-03': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'development-cache-03', 'ref': 'esreekarreddy/vpi-current-development-cache-03', 'suffixes': ['development-cache-03']}, 'development-cache-04': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'development-cache-04', 'ref': 'esreekarreddy/vpi-current-development-cache-04', 'suffixes': ['development-cache-04']}, 'development-cache-05': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'development-cache-05', 'ref': 'esreekarreddy/vpi-current-development-cache-05', 'suffixes': ['development-cache-05']}, 'development-cache-06': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'development-cache-06', 'ref': 'esreekarreddy/vpi-current-development-cache-06', 'suffixes': ['development-cache-06']}, 'development-cache-07': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'development-cache-07', 'ref': 'esreekarreddy/vpi-current-development-cache-07', 'suffixes': ['development-cache-07']}, 'development-cache-08': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'development-cache-08', 'ref': 'esreekarreddy/vpi-current-development-cache-08', 'suffixes': ['development-cache-08']}, 'development-cache-09': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'development-cache-09', 'ref': 'esreekarreddy/vpi-current-development-cache-09', 'suffixes': ['development-cache-09']}, 'development-evaluate': {'contains_outcomes': True, 'kind': 'kernel', 'marker': 'evaluation_lock.json', 'producer_job': 'development-evaluate', 'ref': 'esreekarreddy/vpi-current-development-evaluate', 'suffixes': ['development-evaluate']}, 'development-geometry': {'kind': 'kernel', 'marker': 'geometry_manifest.json', 'producer_job': 'development-geometry', 'ref': 'esreekarreddy/vpi-current-development-geometry', 'suffixes': ['development-geometry']}, 'development-index': {'kind': 'kernel', 'marker': 'rgb_index.json', 'producer_job': 'development-index', 'ref': 'esreekarreddy/vpi-current-development-index', 'suffixes': ['development-index']}, 'development-neutral': {'kind': 'kernel', 'marker': 'neutral_correspondence.json', 'producer_job': 'development-neutral', 'ref': 'esreekarreddy/vpi-current-development-neutral', 'suffixes': ['development-neutral']}, 'development-predict': {'kind': 'kernel', 'marker': 'prediction_lock.json', 'producer_job': 'development-predict', 'ref': 'esreekarreddy/vpi-current-development-predict', 'suffixes': ['development-predict']}, 'development-regions': {'availability': 'existing_kaggle', 'kind': 'kernel', 'marker': 'full-development-lock.json', 'ref': 'esreekarreddy/toysad-8k-frozen-development-pipeline', 'required_sha256': {'full-development-lock.json': '9d8505d2bccdcf9d6e59ebf39c52676278517ff6564c391aa88cb4fb113c050c'}, 'suffixes': ['toysad8k_frozen_development_pipeline']}, 'development-truth': {'contains_outcomes': True, 'kind': 'kernel', 'marker': 'current_evaluator_manifest.json', 'producer_job': 'development-truth', 'ref': 'esreekarreddy/vpi-current-development-truth', 'suffixes': ['development-truth']}, 'freeze': {'kind': 'kernel', 'marker': 'method_freeze.json', 'producer_job': 'freeze', 'ref': 'esreekarreddy/vpi-current-freeze', 'suffixes': ['freeze']}, 'internal_lock-cache': {'kind': 'kernel', 'marker': 'cohort_manifest.json', 'producer_job': 'internal_lock-cache', 'ref': 'esreekarreddy/vpi-current-internal-lock-cache', 'suffixes': ['internal_lock-cache']}, 'internal_lock-cache-00': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'internal_lock-cache-00', 'ref': 'esreekarreddy/vpi-current-internal-lock-cache-00', 'suffixes': ['internal_lock-cache-00']}, 'internal_lock-cache-01': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'internal_lock-cache-01', 'ref': 'esreekarreddy/vpi-current-internal-lock-cache-01', 'suffixes': ['internal_lock-cache-01']}, 'internal_lock-cache-02': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'internal_lock-cache-02', 'ref': 'esreekarreddy/vpi-current-internal-lock-cache-02', 'suffixes': ['internal_lock-cache-02']}, 'internal_lock-cache-03': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'internal_lock-cache-03', 'ref': 'esreekarreddy/vpi-current-internal-lock-cache-03', 'suffixes': ['internal_lock-cache-03']}, 'internal_lock-geometry': {'kind': 'kernel', 'marker': 'geometry_manifest.json', 'producer_job': 'internal_lock-geometry', 'ref': 'esreekarreddy/vpi-current-internal-lock-geometry', 'suffixes': ['internal_lock-geometry']}, 'internal_lock-index': {'kind': 'kernel', 'marker': 'rgb_index.json', 'producer_job': 'internal_lock-index', 'ref': 'esreekarreddy/vpi-current-internal-lock-index', 'suffixes': ['internal_lock-index']}, 'internal_lock-neutral': {'kind': 'kernel', 'marker': 'neutral_correspondence.json', 'producer_job': 'internal_lock-neutral', 'ref': 'esreekarreddy/vpi-current-internal-lock-neutral', 'suffixes': ['internal_lock-neutral']}, 'internal_lock-regions': {'kind': 'kernel', 'marker': 'artifact_lock.json', 'producer_job': 'internal_lock-regions', 'ref': 'esreekarreddy/vpi-current-internal-lock-regions', 'suffixes': ['internal_lock-regions']}, 'internal_lock-regions-00': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'internal_lock-regions-00', 'ref': 'esreekarreddy/vpi-current-internal-lock-regions-00', 'suffixes': ['internal_lock-regions-00']}, 'internal_lock-regions-01': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'internal_lock-regions-01', 'ref': 'esreekarreddy/vpi-current-internal-lock-regions-01', 'suffixes': ['internal_lock-regions-01']}, 'internal_lock-regions-02': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'internal_lock-regions-02', 'ref': 'esreekarreddy/vpi-current-internal-lock-regions-02', 'suffixes': ['internal_lock-regions-02']}, 'internal_lock-regions-03': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'internal_lock-regions-03', 'ref': 'esreekarreddy/vpi-current-internal-lock-regions-03', 'suffixes': ['internal_lock-regions-03']}, 'internal_lock-rgb': {'kind': 'kernel', 'marker': 'package_record.json', 'producer_job': 'internal_lock-rgb', 'ref': 'esreekarreddy/vpi-current-internal-lock-rgb', 'suffixes': ['internal_lock-rgb/model-input']}, 'internal_lock-truth': {'contains_outcomes': True, 'kind': 'kernel', 'marker': 'current_evaluator_manifest.json', 'producer_job': 'internal_lock-truth', 'ref': 'esreekarreddy/vpi-current-internal-lock-truth', 'suffixes': ['internal_lock-truth']}, 'models': {'kind': 'kernel', 'marker': 'model_assets_receipt.json', 'producer_job': 'setup-dino', 'ref': 'esreekarreddy/vpi-current-setup-dino', 'suffixes': ['setup-dino/model-assets']}, 'official-exposure': {'kind': 'kernel', 'marker': 'official_exposure_ledger.json', 'producer_job': 'official-exposure', 'ref': 'esreekarreddy/vpi-current-official-exposure', 'suffixes': ['official-exposure']}, 'raw': {'availability': 'existing_kaggle', 'contains_outcomes': True, 'kind': 'dataset', 'marker': 'package-checksums.sha256', 'ref': 'esreekarreddy/toysad8k-full-private-eda', 'suffixes': ['', 'toysad8k-full-private-eda']}, 'reuse100': {'availability': 'prepared_local_upload', 'kind': 'dataset', 'marker': 'feature_shard_lock.json', 'ref': 'esreekarreddy/vpi-current-development100-feature-reuse', 'suffixes': ['']}, 'rgb-confirm': {'availability': 'existing_kaggle', 'kind': 'dataset', 'marker': 'package_record.json', 'ref': 'esreekarreddy/vpi-toysad-ds2-confirmation-rgb', 'suffixes': ['model-input', '']}, 'rgb-remainder': {'availability': 'existing_kaggle', 'kind': 'dataset', 'marker': 'package_record.json', 'ref': 'esreekarreddy/vpi-toysad-development-remainder-model-input', 'suffixes': ['development-remainder/model-input', 'model-input', '']}, 'rgb-screen': {'availability': 'existing_kaggle', 'kind': 'dataset', 'marker': 'package_record.json', 'ref': 'esreekarreddy/vpi-toysad-ds1-screen100-model-input', 'suffixes': ['model-input', '']}, 'runtime-cpu': {'kind': 'kernel', 'marker': 'runtime_profile_lock.json', 'producer_job': 'setup-cpu', 'ref': 'esreekarreddy/vpi-current-setup-cpu', 'suffixes': ['setup-cpu/runtime-profile']}, 'runtime-dino': {'kind': 'kernel', 'marker': 'runtime_profile_lock.json', 'producer_job': 'setup-dino', 'ref': 'esreekarreddy/vpi-current-setup-dino', 'suffixes': ['setup-dino/runtime-profile']}, 'runtime-regions': {'kind': 'kernel', 'marker': 'runtime_profile_lock.json', 'producer_job': 'setup-regions', 'ref': 'esreekarreddy/vpi-current-setup-regions', 'suffixes': ['setup-regions/runtime-profile']}, 'runtime-tracking': {'kind': 'kernel', 'marker': 'runtime_profile_lock.json', 'producer_job': 'setup-tracking', 'ref': 'esreekarreddy/vpi-current-setup-tracking', 'suffixes': ['setup-tracking/runtime-profile']}, 'seen-cache': {'kind': 'kernel', 'marker': 'cohort_manifest.json', 'producer_job': 'seen-cache', 'ref': 'esreekarreddy/vpi-current-seen-cache', 'suffixes': ['seen-cache']}, 'seen-cache-00': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'seen-cache-00', 'ref': 'esreekarreddy/vpi-current-seen-cache-00', 'suffixes': ['seen-cache-00']}, 'seen-cache-01': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'seen-cache-01', 'ref': 'esreekarreddy/vpi-current-seen-cache-01', 'suffixes': ['seen-cache-01']}, 'seen-cache-02': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'seen-cache-02', 'ref': 'esreekarreddy/vpi-current-seen-cache-02', 'suffixes': ['seen-cache-02']}, 'seen-cache-03': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'seen-cache-03', 'ref': 'esreekarreddy/vpi-current-seen-cache-03', 'suffixes': ['seen-cache-03']}, 'seen-cache-04': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'seen-cache-04', 'ref': 'esreekarreddy/vpi-current-seen-cache-04', 'suffixes': ['seen-cache-04']}, 'seen-cache-05': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'seen-cache-05', 'ref': 'esreekarreddy/vpi-current-seen-cache-05', 'suffixes': ['seen-cache-05']}, 'seen-cache-06': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'seen-cache-06', 'ref': 'esreekarreddy/vpi-current-seen-cache-06', 'suffixes': ['seen-cache-06']}, 'seen-cache-07': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'seen-cache-07', 'ref': 'esreekarreddy/vpi-current-seen-cache-07', 'suffixes': ['seen-cache-07']}, 'seen-geometry': {'kind': 'kernel', 'marker': 'geometry_manifest.json', 'producer_job': 'seen-geometry', 'ref': 'esreekarreddy/vpi-current-seen-geometry', 'suffixes': ['seen-geometry']}, 'seen-index': {'kind': 'kernel', 'marker': 'rgb_index.json', 'producer_job': 'seen-index', 'ref': 'esreekarreddy/vpi-current-seen-index', 'suffixes': ['seen-index']}, 'seen-neutral': {'kind': 'kernel', 'marker': 'neutral_correspondence.json', 'producer_job': 'seen-neutral', 'ref': 'esreekarreddy/vpi-current-seen-neutral', 'suffixes': ['seen-neutral']}, 'seen-regions': {'kind': 'kernel', 'marker': 'artifact_lock.json', 'producer_job': 'seen-regions', 'ref': 'esreekarreddy/vpi-current-seen-regions', 'suffixes': ['seen-regions']}, 'seen-regions-00': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'seen-regions-00', 'ref': 'esreekarreddy/vpi-current-seen-regions-00', 'suffixes': ['seen-regions-00']}, 'seen-regions-01': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'seen-regions-01', 'ref': 'esreekarreddy/vpi-current-seen-regions-01', 'suffixes': ['seen-regions-01']}, 'seen-regions-02': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'seen-regions-02', 'ref': 'esreekarreddy/vpi-current-seen-regions-02', 'suffixes': ['seen-regions-02']}, 'seen-regions-03': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'seen-regions-03', 'ref': 'esreekarreddy/vpi-current-seen-regions-03', 'suffixes': ['seen-regions-03']}, 'seen-regions-04': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'seen-regions-04', 'ref': 'esreekarreddy/vpi-current-seen-regions-04', 'suffixes': ['seen-regions-04']}, 'seen-regions-05': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'seen-regions-05', 'ref': 'esreekarreddy/vpi-current-seen-regions-05', 'suffixes': ['seen-regions-05']}, 'seen-regions-06': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'seen-regions-06', 'ref': 'esreekarreddy/vpi-current-seen-regions-06', 'suffixes': ['seen-regions-06']}, 'seen-regions-07': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'seen-regions-07', 'ref': 'esreekarreddy/vpi-current-seen-regions-07', 'suffixes': ['seen-regions-07']}, 'seen-rgb': {'kind': 'kernel', 'marker': 'package_record.json', 'producer_job': 'seen-rgb', 'ref': 'esreekarreddy/vpi-current-seen-rgb', 'suffixes': ['seen-rgb/model-input']}, 'seen-truth': {'contains_outcomes': True, 'kind': 'kernel', 'marker': 'current_evaluator_manifest.json', 'producer_job': 'seen-truth', 'ref': 'esreekarreddy/vpi-current-seen-truth', 'suffixes': ['seen-truth']}, 'selection': {'availability': 'existing_kaggle', 'kind': 'dataset', 'marker': 'experiment_roles.jsonl', 'ref': 'esreekarreddy/vpi-toysad-selection-private', 'suffixes': ['toysad8k-train-2000-v2']}, 'source': {'availability': 'prepare_upload', 'kind': 'dataset', 'marker': 'prepared_source_manifest.json', 'ref': 'esreekarreddy/vpi-current-evaluation-source', 'suffixes': ['source', '']}, 'support': {'availability': 'prepare_upload', 'kind': 'dataset', 'marker': 'support_manifest.json', 'ref': 'esreekarreddy/vpi-current-evaluation-support', 'suffixes': ['']}, 'unseen-cache': {'kind': 'kernel', 'marker': 'cohort_manifest.json', 'producer_job': 'unseen-cache', 'ref': 'esreekarreddy/vpi-current-unseen-cache', 'suffixes': ['unseen-cache']}, 'unseen-cache-00': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'unseen-cache-00', 'ref': 'esreekarreddy/vpi-current-unseen-cache-00', 'suffixes': ['unseen-cache-00']}, 'unseen-cache-01': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'unseen-cache-01', 'ref': 'esreekarreddy/vpi-current-unseen-cache-01', 'suffixes': ['unseen-cache-01']}, 'unseen-cache-02': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'unseen-cache-02', 'ref': 'esreekarreddy/vpi-current-unseen-cache-02', 'suffixes': ['unseen-cache-02']}, 'unseen-cache-03': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'unseen-cache-03', 'ref': 'esreekarreddy/vpi-current-unseen-cache-03', 'suffixes': ['unseen-cache-03']}, 'unseen-cache-04': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'unseen-cache-04', 'ref': 'esreekarreddy/vpi-current-unseen-cache-04', 'suffixes': ['unseen-cache-04']}, 'unseen-cache-05': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'unseen-cache-05', 'ref': 'esreekarreddy/vpi-current-unseen-cache-05', 'suffixes': ['unseen-cache-05']}, 'unseen-cache-06': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'unseen-cache-06', 'ref': 'esreekarreddy/vpi-current-unseen-cache-06', 'suffixes': ['unseen-cache-06']}, 'unseen-cache-07': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'unseen-cache-07', 'ref': 'esreekarreddy/vpi-current-unseen-cache-07', 'suffixes': ['unseen-cache-07']}, 'unseen-cache-08': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'unseen-cache-08', 'ref': 'esreekarreddy/vpi-current-unseen-cache-08', 'suffixes': ['unseen-cache-08']}, 'unseen-cache-09': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'unseen-cache-09', 'ref': 'esreekarreddy/vpi-current-unseen-cache-09', 'suffixes': ['unseen-cache-09']}, 'unseen-cache-10': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'unseen-cache-10', 'ref': 'esreekarreddy/vpi-current-unseen-cache-10', 'suffixes': ['unseen-cache-10']}, 'unseen-cache-11': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'unseen-cache-11', 'ref': 'esreekarreddy/vpi-current-unseen-cache-11', 'suffixes': ['unseen-cache-11']}, 'unseen-cache-12': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'unseen-cache-12', 'ref': 'esreekarreddy/vpi-current-unseen-cache-12', 'suffixes': ['unseen-cache-12']}, 'unseen-cache-13': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'unseen-cache-13', 'ref': 'esreekarreddy/vpi-current-unseen-cache-13', 'suffixes': ['unseen-cache-13']}, 'unseen-cache-14': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'unseen-cache-14', 'ref': 'esreekarreddy/vpi-current-unseen-cache-14', 'suffixes': ['unseen-cache-14']}, 'unseen-cache-15': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'unseen-cache-15', 'ref': 'esreekarreddy/vpi-current-unseen-cache-15', 'suffixes': ['unseen-cache-15']}, 'unseen-cache-16': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'unseen-cache-16', 'ref': 'esreekarreddy/vpi-current-unseen-cache-16', 'suffixes': ['unseen-cache-16']}, 'unseen-cache-17': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'unseen-cache-17', 'ref': 'esreekarreddy/vpi-current-unseen-cache-17', 'suffixes': ['unseen-cache-17']}, 'unseen-cache-18': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'unseen-cache-18', 'ref': 'esreekarreddy/vpi-current-unseen-cache-18', 'suffixes': ['unseen-cache-18']}, 'unseen-cache-19': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'unseen-cache-19', 'ref': 'esreekarreddy/vpi-current-unseen-cache-19', 'suffixes': ['unseen-cache-19']}, 'unseen-cache-20': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'unseen-cache-20', 'ref': 'esreekarreddy/vpi-current-unseen-cache-20', 'suffixes': ['unseen-cache-20']}, 'unseen-cache-21': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'unseen-cache-21', 'ref': 'esreekarreddy/vpi-current-unseen-cache-21', 'suffixes': ['unseen-cache-21']}, 'unseen-cache-22': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'unseen-cache-22', 'ref': 'esreekarreddy/vpi-current-unseen-cache-22', 'suffixes': ['unseen-cache-22']}, 'unseen-cache-23': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'unseen-cache-23', 'ref': 'esreekarreddy/vpi-current-unseen-cache-23', 'suffixes': ['unseen-cache-23']}, 'unseen-cache-24': {'kind': 'kernel', 'marker': 'cache_shard_lock.json', 'producer_job': 'unseen-cache-24', 'ref': 'esreekarreddy/vpi-current-unseen-cache-24', 'suffixes': ['unseen-cache-24']}, 'unseen-geometry': {'kind': 'kernel', 'marker': 'geometry_manifest.json', 'producer_job': 'unseen-geometry', 'ref': 'esreekarreddy/vpi-current-unseen-geometry', 'suffixes': ['unseen-geometry']}, 'unseen-index': {'kind': 'kernel', 'marker': 'rgb_index.json', 'producer_job': 'unseen-index', 'ref': 'esreekarreddy/vpi-current-unseen-index', 'suffixes': ['unseen-index']}, 'unseen-neutral': {'kind': 'kernel', 'marker': 'neutral_correspondence.json', 'producer_job': 'unseen-neutral', 'ref': 'esreekarreddy/vpi-current-unseen-neutral', 'suffixes': ['unseen-neutral']}, 'unseen-regions': {'kind': 'kernel', 'marker': 'artifact_lock.json', 'producer_job': 'unseen-regions', 'ref': 'esreekarreddy/vpi-current-unseen-regions', 'suffixes': ['unseen-regions']}, 'unseen-regions-00': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'unseen-regions-00', 'ref': 'esreekarreddy/vpi-current-unseen-regions-00', 'suffixes': ['unseen-regions-00']}, 'unseen-regions-01': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'unseen-regions-01', 'ref': 'esreekarreddy/vpi-current-unseen-regions-01', 'suffixes': ['unseen-regions-01']}, 'unseen-regions-02': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'unseen-regions-02', 'ref': 'esreekarreddy/vpi-current-unseen-regions-02', 'suffixes': ['unseen-regions-02']}, 'unseen-regions-03': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'unseen-regions-03', 'ref': 'esreekarreddy/vpi-current-unseen-regions-03', 'suffixes': ['unseen-regions-03']}, 'unseen-regions-04': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'unseen-regions-04', 'ref': 'esreekarreddy/vpi-current-unseen-regions-04', 'suffixes': ['unseen-regions-04']}, 'unseen-regions-05': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'unseen-regions-05', 'ref': 'esreekarreddy/vpi-current-unseen-regions-05', 'suffixes': ['unseen-regions-05']}, 'unseen-regions-06': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'unseen-regions-06', 'ref': 'esreekarreddy/vpi-current-unseen-regions-06', 'suffixes': ['unseen-regions-06']}, 'unseen-regions-07': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'unseen-regions-07', 'ref': 'esreekarreddy/vpi-current-unseen-regions-07', 'suffixes': ['unseen-regions-07']}, 'unseen-regions-08': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'unseen-regions-08', 'ref': 'esreekarreddy/vpi-current-unseen-regions-08', 'suffixes': ['unseen-regions-08']}, 'unseen-regions-09': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'unseen-regions-09', 'ref': 'esreekarreddy/vpi-current-unseen-regions-09', 'suffixes': ['unseen-regions-09']}, 'unseen-regions-10': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'unseen-regions-10', 'ref': 'esreekarreddy/vpi-current-unseen-regions-10', 'suffixes': ['unseen-regions-10']}, 'unseen-regions-11': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'unseen-regions-11', 'ref': 'esreekarreddy/vpi-current-unseen-regions-11', 'suffixes': ['unseen-regions-11']}, 'unseen-regions-12': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'unseen-regions-12', 'ref': 'esreekarreddy/vpi-current-unseen-regions-12', 'suffixes': ['unseen-regions-12']}, 'unseen-regions-13': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'unseen-regions-13', 'ref': 'esreekarreddy/vpi-current-unseen-regions-13', 'suffixes': ['unseen-regions-13']}, 'unseen-regions-14': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'unseen-regions-14', 'ref': 'esreekarreddy/vpi-current-unseen-regions-14', 'suffixes': ['unseen-regions-14']}, 'unseen-regions-15': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'unseen-regions-15', 'ref': 'esreekarreddy/vpi-current-unseen-regions-15', 'suffixes': ['unseen-regions-15']}, 'unseen-regions-16': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'unseen-regions-16', 'ref': 'esreekarreddy/vpi-current-unseen-regions-16', 'suffixes': ['unseen-regions-16']}, 'unseen-regions-17': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'unseen-regions-17', 'ref': 'esreekarreddy/vpi-current-unseen-regions-17', 'suffixes': ['unseen-regions-17']}, 'unseen-regions-18': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'unseen-regions-18', 'ref': 'esreekarreddy/vpi-current-unseen-regions-18', 'suffixes': ['unseen-regions-18']}, 'unseen-regions-19': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'unseen-regions-19', 'ref': 'esreekarreddy/vpi-current-unseen-regions-19', 'suffixes': ['unseen-regions-19']}, 'unseen-regions-20': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'unseen-regions-20', 'ref': 'esreekarreddy/vpi-current-unseen-regions-20', 'suffixes': ['unseen-regions-20']}, 'unseen-regions-21': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'unseen-regions-21', 'ref': 'esreekarreddy/vpi-current-unseen-regions-21', 'suffixes': ['unseen-regions-21']}, 'unseen-regions-22': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'unseen-regions-22', 'ref': 'esreekarreddy/vpi-current-unseen-regions-22', 'suffixes': ['unseen-regions-22']}, 'unseen-regions-23': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'unseen-regions-23', 'ref': 'esreekarreddy/vpi-current-unseen-regions-23', 'suffixes': ['unseen-regions-23']}, 'unseen-regions-24': {'kind': 'kernel', 'marker': 'region_shard_lock.json', 'producer_job': 'unseen-regions-24', 'ref': 'esreekarreddy/vpi-current-unseen-regions-24', 'suffixes': ['unseen-regions-24']}, 'unseen-rgb': {'kind': 'kernel', 'marker': 'package_record.json', 'producer_job': 'unseen-rgb', 'ref': 'esreekarreddy/vpi-current-unseen-rgb', 'suffixes': ['unseen-rgb/model-input']}, 'unseen-truth': {'contains_outcomes': True, 'kind': 'kernel', 'marker': 'current_evaluator_manifest.json', 'producer_job': 'unseen-truth', 'ref': 'esreekarreddy/vpi-current-unseen-truth', 'suffixes': ['unseen-truth']}, 'wheels-cpu': {'kind': 'kernel', 'marker': 'wheelhouse_manifest.json', 'producer_job': 'setup-cpu', 'ref': 'esreekarreddy/vpi-current-setup-cpu', 'suffixes': ['setup-cpu/wheelhouse']}, 'wheels-dino': {'kind': 'kernel', 'marker': 'wheelhouse_manifest.json', 'producer_job': 'setup-dino', 'ref': 'esreekarreddy/vpi-current-setup-dino', 'suffixes': ['setup-dino/wheelhouse']}, 'wheels-regions': {'kind': 'kernel', 'marker': 'wheelhouse_manifest.json', 'producer_job': 'setup-regions', 'ref': 'esreekarreddy/vpi-current-setup-regions', 'suffixes': ['setup-regions/wheelhouse']}, 'wheels-tracking': {'kind': 'kernel', 'marker': 'wheelhouse_manifest.json', 'producer_job': 'setup-tracking', 'ref': 'esreekarreddy/vpi-current-setup-tracking', 'suffixes': ['setup-tracking/wheelhouse']}}, 'schedule_count': 19, 'schema': 'vpi-current-kaggle-launch-plan/v1', 'scientific_runs_started': False}
    configure_runtime(LAUNCH_PLAN, 'freeze', CODE_ROOT)
    LAUNCH_BINDING = resolve_job(LAUNCH_PLAN, 'freeze')
    import uuid
    _profile_relative = os.environ.get("VPI_BASE_RUNTIME_PROFILE") or "requirements/current-evaluation-runtime.json"
    _attempt = uuid.uuid4().hex[:12]
    _target = _runtime.runtime_target("/tmp/vpi-runtime-" + 'freeze' + "-" + _attempt)
    os.environ["VPI_PACKAGE_TARGET"] = str(_target)
    _isolated_profile = Path("/kaggle/working/runtime-profiles") / ('freeze' + "-" + _attempt)
    _prepare_command = [sys.executable, str(CODE_ROOT / "scripts/prepare_runtime_environment.py"),
                        "--profile", str(CODE_ROOT / _profile_relative),
                        "--wheelhouse", os.environ["VPI_WHEELHOUSE"],
                        "--target", str(_target), "--output-root", str(_isolated_profile)]
    import subprocess
    subprocess.run(_prepare_command, check=True, timeout=1020)
    _base_profile = json.loads((CODE_ROOT / _profile_relative).read_text())
    _base_contract = _base_profile.get("runtime_contract", _base_profile)
    _runtime.observed_contract(_isolated_profile, CODE_ROOT / _profile_relative, _base_contract)
    _dependency_receipt = json.loads((_target / "dependency_receipt.json").read_text())
    if _dependency_receipt.get("runtime_contract") != _base_contract:
        raise ValueError("Fresh isolated dependency target differs from the registered profile")
    _child_path = Path("/tmp") / ("vpi-current-job-" + 'freeze' + ".py")
    _child_path.write_text('from pathlib import Path\nimport json\nimport os\nimport sys\n\nCODE_ROOT = Path(os.environ["VPI_RESOLVED_CODE_ROOT"]).resolve(strict=True)\nNOTEBOOK_RELATIVE = \'notebooks/input-preparation/current-kaggle-preparation.ipynb\'\nRUNTIME_PROFILE = None\nsys.path.insert(0, str(CODE_ROOT / "scripts"))\nsys.path.insert(0, str(CODE_ROOT / "src"))\nimport current_notebook_runtime as _runtime\nBOOTSTRAP = _runtime.bootstrap(CODE_ROOT, NOTEBOOK_RELATIVE, runtime_profile=RUNTIME_PROFILE)\nCODE_PROVENANCE = BOOTSTRAP["provenance"]\nROOT = CODE_ROOT\n\ndef display(value):\n    print(json.dumps(value, sort_keys=True, default=str))\n\nCONFIG = _runtime.resolve_config(\'preparation\', ROOT)\nCONFIG[\'output_root\'] = str(_runtime.writable_output(CONFIG[\'output_root\']))\nfrom current_stage_execution import run_stage\nACTION = CONFIG[\'action\']\n\ndef prepare():\n    if ACTION == \'freeze\':\n        from vpi_next_evaluation.contracts import prepare_method_freeze\n        return prepare_method_freeze(CONFIG[\'reference_root\'], CONFIG[\'output_root\'], ROOT)\n    if ACTION in {\'rgb_index\',\'rgb_release\'}:\n        from vpi_next_evaluation.native_rgb import prepare_rgb_index, release_rgb\n        return (prepare_rgb_index if ACTION==\'rgb_index\' else release_rgb)(CONFIG,ROOT)\n    if ACTION in {\'raw_geometry\',\'raw_truth\'}:\n        from vpi_next_evaluation.raw_reference import prepare_raw_geometry, convert_raw_evaluator\n        return (prepare_raw_geometry if ACTION==\'raw_geometry\' else convert_raw_evaluator)(CONFIG)\n    if ACTION == \'neutral\':\n        from vpi_next_evaluation.native_evaluator import prepare_neutral\n        return prepare_neutral(CONFIG)\n    if ACTION == \'cache_merge\':\n        from vpi_next_evaluation.cache_shards import merge_cache_shards\n        return merge_cache_shards(CONFIG,ROOT)\n    if ACTION == \'region_merge\':\n        from vpi_next_evaluation.region_shards import merge_region_shards\n        return merge_region_shards(CONFIG,ROOT)\n    if ACTION == \'prepare_shard\':\n        from vpi_next_evaluation.native_cache import prepare_shard\n        return prepare_shard(CONFIG,ROOT)\n    if ACTION == \'official_exposure\':\n        from vpi_next_evaluation.exposure import prepare_official_exposure\n        return prepare_official_exposure(CONFIG)\n    raise ValueError(\'Unknown prepared launch operation\')\n\nRESULT = run_stage(\'preparation-\'+ACTION,CONFIG,prepare,CODE_PROVENANCE)\ndisplay(RESULT)\n')
    _environment = dict(os.environ)
    _environment["VPI_RESOLVED_CODE_ROOT"] = str(CODE_ROOT)
    _environment["PYTHONPATH"] = os.pathsep.join([str(_target), str(CODE_ROOT / "scripts"), str(CODE_ROOT / "src")])
    _completed = subprocess.run([sys.executable, "-B", str(_child_path)], env=_environment, check=True, timeout=7200)
    JOB_COMPLETION = complete_job(LAUNCH_PLAN, 'freeze', _manifest["snapshot_sha256"])
    print(json.dumps({"launch_binding": LAUNCH_BINDING, "child_returncode": _completed.returncode, "completion": JOB_COMPLETION}, sort_keys=True, default=str))
except BaseException as _bootstrap_error:
    _failure_root = Path('/kaggle/working/runtime-receipts') if Path('/kaggle').is_dir() else Path.cwd()/'.local/runtime-receipts'
    _failure_root.mkdir(parents=True, exist_ok=True)
    _failure = {'status':'bootstrap_failed','error_type':type(_bootstrap_error).__name__,'scientific_run_started':False}
    _temporary = _failure_root/'bootstrap-failure.json.partial'
    _temporary.write_text(json.dumps(_failure,sort_keys=True)+'\n')
    os.replace(_temporary,_failure_root/'bootstrap-failure.json')
    raise

{
  "status": "runtime_profile_prepared",
  "python": "3.12.13",
  "profile_sha256": "b43d05c002fcc11da3e0a724e35082c61483927932941824a202afe3b3f8279d",
  "private_target_prepared": true,
  "models_loaded": false,
  "datasets_opened": false,
  "scientific_methods_executed": false
}
{"assisted": {"fusion": "top2", "normalization": "within-view percentile", "scorer": "foreground patch bank", "tail_fraction": 0.02, "target_identity_exclusion": "all selected views"}, "assisted_provenance": ["dataset_reference", "user_confirmed"], "automatic": {"association_distance": 0.2, "selector": "forward/P1/P2 percentile consensus"}, "bootstrap": {"samples": 2000, "seed": 32040, "unit": "asset_group"}, "claim": "relative object ranking; no calibrated probability or independent confirmation implied", "methods": {"assisted": "object_bank_top2", "automatic": "winner_cluster_consensus"}, "metrics": ["top1_tie_expected", "recall_at_2_tie_expected", "ap_tie_expected", "reciprocal_rank_tie_expected"], "no_re